In [2]:
# =========================================================
# FRAMEWORK MANUAL — BASE ESTRUTURAL
# IMDb + Hubara como primeiro caso
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import pandas as pd


# ---------------------------------------------------------
# 1) Especificação do cenário
# ---------------------------------------------------------
# Representa o "mundo" que será testado:
# - nome do dataset
# - entidades
# - relacionamentos
# - queries do workload
#
# Exemplo:
# dataset = IMDb
# entities = User, Movie, Rating, ...
@dataclass
class ScenarioSpec:
    name: str
    description: str
    entities: List[str]
    relationships: List[Dict[str, Any]]
    workload_queries: List[Dict[str, Any]]


# ---------------------------------------------------------
# 2) Fragmento recomendado
# ---------------------------------------------------------
# Cada fragmento é um grupo de entidades que será alocado
# em um tipo/modelo de banco.
#
# Exemplo:
# Fragmento 1 = ['User'] -> relational
# Fragmento 2 = ['Movie', 'Genre', ...] -> column
@dataclass
class FragmentSpec:
    name: str
    entities: List[str]
    model: str   # relational, document, column, graph, etc.
    notes: str = ""


# ---------------------------------------------------------
# 3) Recomendação final
# ---------------------------------------------------------
# Representa a saída final de um método como o Hubara.
#
# Exemplo:
# recommendation_name = "Hubara IMDb"
# fragments = [FragmentSpec(...), FragmentSpec(...)]
@dataclass
class RecommendationSpec:
    recommendation_name: str
    source_method: str
    source_dataset: str
    fragments: List[FragmentSpec]


# ---------------------------------------------------------
# 4) Banco físico concreto
# ---------------------------------------------------------
# Aqui diferenciamos "modelo" de "produto".
#
# Exemplo:
# model = relational
# engine = PostgreSQL
#
# ou:
# model = document
# engine = MongoDB
@dataclass
class PhysicalDBSpec:
    model: str
    engine: str
    host: str = "localhost"
    port: Optional[int] = None
    database_name: Optional[str] = None
    username: Optional[str] = None
    password: Optional[str] = None

    # opcionais, úteis para bancos como MongoDB
    connection_uri: Optional[str] = None
    options: Dict[str, Any] = field(default_factory=dict)


# ---------------------------------------------------------
# 5) Plano de materialização
# ---------------------------------------------------------
# Traduz a recomendação abstrata para bancos concretos.
#
# Exemplo:
# fragment User -> PostgreSQL
# fragment Content -> Cassandra
@dataclass
class MaterializationPlan:
    scenario_name: str
    recommendation_name: str
    fragment_to_db: Dict[str, PhysicalDBSpec]


# ---------------------------------------------------------
# 6) Especificação de uma query do benchmark
# ---------------------------------------------------------
# Aqui vamos guardar:
# - nome da query
# - tipo
# - entidades tocadas
# - texto abstrato
#
# Mais tarde, podemos ter versões por banco:
# sql_text, mongo_text, cypher_text, etc.
@dataclass
class BenchmarkQuery:
    name: str
    query_type: str
    entities_involved: List[str]
    abstract_query: str
    expected_fragment: Optional[str] = None
    expected_db_model: Optional[str] = None


# ---------------------------------------------------------
# 7) Especificação do workload
# ---------------------------------------------------------
@dataclass
class WorkloadSpec:
    name: str
    description: str
    queries: List[BenchmarkQuery]
    repetitions: int = 10


# ---------------------------------------------------------
# 8) Resultado de execução de uma query
# ---------------------------------------------------------
@dataclass
class QueryExecutionResult:
    query_name: str
    fragment_name: str
    db_engine: str
    run_id: int
    benchmark_phase: str   # "cold" ou "hot"
    latency_ms: float
    success: bool
    error_message: Optional[str] = None


# ---------------------------------------------------------
# 9) Resultado consolidado do benchmark
# ---------------------------------------------------------
@dataclass
class BenchmarkResult:
    scenario_name: str
    recommendation_name: str
    query_results: List[QueryExecutionResult] = field(default_factory=list)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "scenario_name": self.scenario_name,
                "recommendation_name": self.recommendation_name,
                "query_name": r.query_name,
                "fragment_name": r.fragment_name,
                "db_engine": r.db_engine,
                "run_id": r.run_id,
                "benchmark_phase": r.benchmark_phase,
                "latency_ms": r.latency_ms,
                "success": r.success,
                "error_message": r.error_message,
            }
            for r in self.query_results
        ])

In [3]:
# =========================================================
# INSTÂNCIA 1 — CENÁRIO IMDb + RECOMENDAÇÃO HUBARA
# =========================================================

# ---------------------------------------------------------
# 1) Cenário IMDb
# ---------------------------------------------------------
imdb_scenario = ScenarioSpec(
    name="IMDb",
    description="IMDb-like scenario based on the Hubara paper",
    entities=[
        "Person",
        "User",
        "Genre",
        "Role",
        "Rate",
        "WatchItem",
        "Series",
        "Movie",
        "Episode",
    ],
    relationships=[
        {"source": "Person", "target": "Role", "name": "acts_in_via_role"},
        {"source": "Role", "target": "WatchItem", "name": "role_of_watchitem"},
        {"source": "Person", "target": "WatchItem", "name": "directs"},
        {"source": "Person", "target": "WatchItem", "name": "produces"},
        {"source": "Genre", "target": "WatchItem", "name": "has_genre"},
        {"source": "User", "target": "Rate", "name": "rates"},
        {"source": "Rate", "target": "WatchItem", "name": "rate_of_watchitem"},
        {"source": "Series", "target": "WatchItem", "name": "series_is_watchitem"},
        {"source": "Movie", "target": "WatchItem", "name": "movie_is_watchitem"},
        {"source": "Episode", "target": "WatchItem", "name": "episode_is_watchitem"},
        {"source": "Series", "target": "Episode", "name": "series_contains_episode"},
    ],
    workload_queries=[
        {
            "name": "Q1_Login",
            "type": "select",
            "entities": ["User"],
            "abstract_query": "RETURN password FROM User WHERE username = ?"
        },
        {
            "name": "Q2_SimpleSearch",
            "type": "select",
            "entities": ["WatchItem"],
            "abstract_query": "RETURN ALL FROM WatchItem WHERE title = ?"
        },
        {
            "name": "Q3_AddEntitiesAndRelationships",
            "type": "insert",
            "entities": ["Person", "WatchItem", "Role"],
            "abstract_query": "INSERT Person and connect Person with WatchItem"
        },
        {
            "name": "Q4_Recommendation",
            "type": "select",
            "entities": ["WatchItem", "Genre", "Movie", "Series", "Episode"],
            "abstract_query": "Recommendation query by genre and type"
        },
        {
            "name": "Q5_AllPersonsOfTypeForWatchItem",
            "type": "select",
            "entities": ["WatchItem", "Person", "Role"],
            "abstract_query": "RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor"
        },
    ]
)


# ---------------------------------------------------------
# 2) Recomendação Hubara para IMDb
# ---------------------------------------------------------
# Resultado que você reproduziu:
# - Fragmento de conteúdo -> Column
# - Fragmento de User -> RDBMS
hubara_imdb_recommendation = RecommendationSpec(
    recommendation_name="Hubara_IMDb_Recommendation",
    source_method="Hubara",
    source_dataset="IMDb",
    fragments=[
        FragmentSpec(
            name="ContentFragment",
            entities=["Episode", "Genre", "Movie", "Person", "Rate", "Role", "Series", "WatchItem"],
            model="column",
            notes="Recommended by the implemented Hubara workflow"
        ),
        FragmentSpec(
            name="UserFragment",
            entities=["User"],
            model="relational",
            notes="Recommended by the implemented Hubara workflow"
        ),
    ]
)


# ---------------------------------------------------------
# 3) Plano físico concreto
# ---------------------------------------------------------
# Aqui você escolhe quais produtos concretos representam
# cada modelo lógico.
#
# Exemplo inicial:
# relational -> PostgreSQL
# column -> Cassandra
#
# Você pode trocar Cassandra por outro depois.
hubara_imdb_materialization = MaterializationPlan(
    scenario_name="IMDb",
    recommendation_name="Hubara_IMDb_Recommendation",
    fragment_to_db={
        "UserFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,
            database_name="imdb_user_db",
            username="postgres",
            password="postgres"
        ),
        "ContentFragment": PhysicalDBSpec(
            model="column",
            engine="Cassandra",
            host="127.0.0.1",
            port=59042,
            database_name="imdb_content_db"
        ),
    }
)


# =========================================================
# BLOCO M1 — MATERIALIZAÇÃO ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

hubara_imdb_mongo_materialization = MaterializationPlan(
    scenario_name="IMDb",
    recommendation_name="Hubara_IMDb_Recommendation_MongoAlternative",
    fragment_to_db={
        "UserFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,   # via túnel SSH, como você já configurou
            database_name="imdb_user_db_mongo_alt",
            username="postgres",
            password="postgres"
        ),
        "ContentFragment": PhysicalDBSpec(
            model="document",
            engine="MongoDB",
            host="127.0.0.1",
            port=57017,
            database_name="imdb_content_mongo_db",
            username="mongo",
            password="mongo"
        ),
    }
)

print("Materialização alternativa criada com sucesso.")
print(hubara_imdb_mongo_materialization)


# ---------------------------------------------------------
# 4) Workload do benchmark
# ---------------------------------------------------------
imdb_workload = WorkloadSpec(
    name="IMDb_Hubara_Workload",
    description="Benchmark workload for the IMDb scenario using the Hubara recommendation",
    queries=[
        BenchmarkQuery(
            name="Q1_Login",
            query_type="select",
            entities_involved=["User"],
            abstract_query="RETURN password FROM User WHERE username = ?",
            expected_fragment="UserFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q2_SimpleSearch",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q3_AddEntitiesAndRelationships",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person and connect Person with WatchItem",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q4_Recommendation",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query="Recommendation query by genre and type",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q5_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Person", "Role"],
            abstract_query="RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
    ],
    repetitions=10
)


# ---------------------------------------------------------
# 4) Workload do benchmark — alternativa MongoDB
# ---------------------------------------------------------
imdb_workload_mongo = WorkloadSpec(
    name="IMDb_Hubara_Workload_Mongo",
    description="Benchmark workload for the IMDb scenario using the Hubara recommendation with MongoDB for the content fragment",
    queries=[
        BenchmarkQuery(
            name="Q1_Login",
            query_type="select",
            entities_involved=["User"],
            abstract_query="RETURN password FROM User WHERE username = ?",
            expected_fragment="UserFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q2_SimpleSearch",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q3_AddEntitiesAndRelationships",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person and connect Person with WatchItem",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q4_Recommendation",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query="Recommendation query by genre and type",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q5_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Person", "Role"],
            abstract_query="RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
    ],
    repetitions=10
)

Materialização alternativa criada com sucesso.
MaterializationPlan(scenario_name='IMDb', recommendation_name='Hubara_IMDb_Recommendation_MongoAlternative', fragment_to_db={'UserFragment': PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_user_db_mongo_alt', username='postgres', password='postgres', connection_uri=None, options={}), 'ContentFragment': PhysicalDBSpec(model='document', engine='MongoDB', host='127.0.0.1', port=57017, database_name='imdb_content_mongo_db', username='mongo', password='mongo', connection_uri=None, options={})})


In [4]:
# =========================================================
# BLOCO V0 — PREPARAR O WORKLOAD CONCEITUAL
# =========================================================
#INSTANCE R(Qi) = quantity entities touched

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "imdb_scenario" not in globals():
    raise NameError(
        "O objeto 'imdb_scenario' não está definido. "
        "Rode primeiro o bloco que cria o cenário IMDb."
    )

# ---------------------------------------------------------
# 2) Transformar o workload do cenário em DataFrame
# ---------------------------------------------------------
workload_conceptual_df = pd.DataFrame(imdb_scenario.workload_queries).copy()

# Renomeia colunas para deixar mais explícito
workload_conceptual_df = workload_conceptual_df.rename(columns={
    "name": "query_name",
    "type": "query_type",
    "entities": "entities_touched"
})

# Garante uma coluna com quantidade de entidades tocadas
workload_conceptual_df["n_entities_touched"] = workload_conceptual_df["entities_touched"].apply(len)

print("Workload conceitual do IMDb:")
display(workload_conceptual_df)

Workload conceitual do IMDb:


,query_name,query_type,entities_touched,abstract_query,n_entities_touched
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3


In [5]:
# =========================================================
# BLOCO V1 — ORGANIZAR OS RELACIONAMENTOS CONCEITUAIS
# =========================================================
#Organize the conceptual relationships

from collections import defaultdict

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "imdb_scenario" not in globals():
    raise NameError(
        "O objeto 'imdb_scenario' não está definido. "
        "Rode primeiro o bloco que cria o cenário IMDb."
    )

# ---------------------------------------------------------
# 2) DataFrame com os relacionamentos conceituais
# ---------------------------------------------------------
conceptual_relationships_df = pd.DataFrame(imdb_scenario.relationships).copy()

# Cria um identificador textual da aresta conceitual
conceptual_relationships_df["edge_id"] = conceptual_relationships_df.apply(
    lambda row: f'{row["source"]} -- {row["target"]} [{row["name"]}]',
    axis=1
)

print("Relacionamentos conceituais do esquema IMDb:")
display(conceptual_relationships_df)

# ---------------------------------------------------------
# 3) Lista de arestas em formato Python
# ---------------------------------------------------------
conceptual_edges = conceptual_relationships_df[
    ["source", "target", "name", "edge_id"]
].to_dict(orient="records")

# ---------------------------------------------------------
# 4) Grafo não-direcionado simples para navegação conceitual
# Observação:
# para calcular conectividade conceitual entre entidades,
# vamos tratar os relacionamentos como não-direcionados.
# ---------------------------------------------------------
conceptual_adj = defaultdict(list)

for edge in conceptual_edges:
    src = edge["source"]
    tgt = edge["target"]
    conceptual_adj[src].append((tgt, edge["name"], edge["edge_id"]))
    conceptual_adj[tgt].append((src, edge["name"], edge["edge_id"]))

print("Entidades presentes no grafo conceitual:")
print(sorted(conceptual_adj.keys()))

Relacionamentos conceituais do esquema IMDb:


,source,target,name,edge_id
0,Person,Role,acts_in_via_role,Person -- Role [acts_in_via_role]
1,Role,WatchItem,role_of_watchitem,Role -- WatchItem [role_of_watchitem]
2,Person,WatchItem,directs,Person -- WatchItem [directs]
3,Person,WatchItem,produces,Person -- WatchItem [produces]
4,Genre,WatchItem,has_genre,Genre -- WatchItem [has_genre]
5,User,Rate,rates,User -- Rate [rates]
6,Rate,WatchItem,rate_of_watchitem,Rate -- WatchItem [rate_of_watchitem]
7,Series,WatchItem,series_is_watchitem,Series -- WatchItem [series_is_watchitem]
8,Movie,WatchItem,movie_is_watchitem,Movie -- WatchItem [movie_is_watchitem]
9,Episode,WatchItem,episode_is_watchitem,Episode -- WatchItem [episode_is_watchitem]


Entidades presentes no grafo conceitual:
['Episode', 'Genre', 'Movie', 'Person', 'Rate', 'Role', 'Series', 'User', 'WatchItem']


In [6]:
# =========================================================
# BLOCO V2 — EXTRAIR Rc(Qi) EXATAMENTE
# =========================================================
# To EXTRACT Rc(Qi) exactily

from itertools import combinations
from collections import deque

# ---------------------------------------------------------
# 1) Função auxiliar:
# verifica se um subconjunto de arestas conecta todas as
# entidades terminais de uma query
# ---------------------------------------------------------
def terminals_are_connected(selected_edges, terminals):
    """
    Retorna True se todas as entidades 'terminals' estiverem
    conectadas no subgrafo formado por 'selected_edges'.
    """
    terminals = list(dict.fromkeys(terminals))  # remove duplicatas preservando ordem

    if len(terminals) <= 1:
        return True

    # monta adjacência apenas com as arestas selecionadas
    adj = defaultdict(set)
    for edge in selected_edges:
        adj[edge["source"]].add(edge["target"])
        adj[edge["target"]].add(edge["source"])

    # busca em largura a partir do primeiro terminal
    start = terminals[0]
    visited = set([start])
    queue = deque([start])

    while queue:
        current = queue.popleft()
        for neighbor in adj[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    # todos os terminais precisam estar no mesmo componente
    return all(t in visited for t in terminals)

# ---------------------------------------------------------
# 2) Função principal:
# calcula o número mínimo de relacionamentos conceituais
# necessários para conectar as entidades da query
# ---------------------------------------------------------
def extract_rc_for_query(terminals, conceptual_edges):
    """
    Retorna um dicionário com:
    - rc: número mínimo de relacionamentos conceituais
    - selected_edge_ids: identificadores das arestas escolhidas
    """
    terminals = list(dict.fromkeys(terminals))

    # Se a query toca 0 ou 1 entidade, não há relacionamento a percorrer
    if len(terminals) <= 1:
        return {
            "rc": 0,
            "selected_edge_ids": []
        }

    # Testa subconjuntos de arestas por tamanho crescente.
    # O primeiro que conectar todos os terminais já é ótimo.
    for k in range(1, len(conceptual_edges) + 1):
        for subset in combinations(conceptual_edges, k):
            nodes_in_subset = set()
            for edge in subset:
                nodes_in_subset.add(edge["source"])
                nodes_in_subset.add(edge["target"])

            # poda simples: se faltam terminais, nem precisa testar conectividade
            if not set(terminals).issubset(nodes_in_subset):
                continue

            if terminals_are_connected(subset, terminals):
                return {
                    "rc": k,
                    "selected_edge_ids": [edge["edge_id"] for edge in subset]
                }

    # Se nada conectar, devolve None (não esperado neste cenário)
    return {
        "rc": None,
        "selected_edge_ids": []
    }

# ---------------------------------------------------------
# 3) Aplicar para cada query do workload
# ---------------------------------------------------------
rc_rows = []

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    terminals = row["entities_touched"]

    rc_info = extract_rc_for_query(terminals, conceptual_edges)

    rc_rows.append({
        "query_name": query_name,
        "query_type": row["query_type"],
        "entities_touched": terminals,
        "n_entities_touched": len(terminals),
        "Rc": rc_info["rc"],
        "Rc_selected_edges": rc_info["selected_edge_ids"]
    })

rc_df = pd.DataFrame(rc_rows)

print("Extração de Rc(Qi):")
display(rc_df)

#Rc -->  relationship number  of query
#Rc(Qi, Er, D) --> relationship number in time execution
# ΔR: how much was reduced. (diferrence between RC and Rc(Qi, Er, D)


Extração de Rc(Qi):


,query_name,query_type,entities_touched,n_entities_touched,Rc,Rc_selected_edges
0,Q1_Login,select,[User],1,0,[]
1,Q2_SimpleSearch,select,[WatchItem],1,0,[]
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa..."
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",5,4,"[Genre -- WatchItem [has_genre], Series -- Wat..."
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa..."


In [7]:
# =========================================================
# BLOCO V3 — CANDIDATOS INICIAIS DE RAIZ er
# =========================================================

from collections import deque

# ---------------------------------------------------------
# 1) Menor distância entre duas entidades no grafo conceitual
# ---------------------------------------------------------
def shortest_entity_distance(source_entity, target_entity, adj):
    """
    Retorna a menor distância (em número de arestas conceituais)
    entre duas entidades do esquema.
    """
    if source_entity == target_entity:
        return 0

    visited = set([source_entity])
    queue = deque([(source_entity, 0)])

    while queue:
        current, dist = queue.popleft()

        for neighbor, rel_name, edge_id in adj[current]:
            if neighbor == target_entity:
                return dist + 1

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))

    return None

# ---------------------------------------------------------
# 2) Heurística de score para raiz:
# - menor distância média até as outras entidades da query
# - menor distância máxima
# - maior número de vizinhos diretos entre entidades da query
# ---------------------------------------------------------
def extract_root_candidates(terminals, adj):
    """
    Retorna um DataFrame ranqueando candidatos a raiz.
    """
    terminals = list(dict.fromkeys(terminals))
    rows = []

    for candidate in terminals:
        distances = []
        direct_links = 0

        for other in terminals:
            if other == candidate:
                continue

            d = shortest_entity_distance(candidate, other, adj)
            distances.append(d)

            if d == 1:
                direct_links += 1

        avg_distance = sum(distances) / len(distances) if distances else 0.0
        max_distance = max(distances) if distances else 0

        rows.append({
            "candidate_root": candidate,
            "avg_distance_to_other_touched_entities": avg_distance,
            "max_distance_to_other_touched_entities": max_distance,
            "direct_links_to_other_touched_entities": direct_links
        })

    root_df = pd.DataFrame(rows).sort_values(
        by=[
            "avg_distance_to_other_touched_entities",
            "max_distance_to_other_touched_entities",
            "direct_links_to_other_touched_entities"
        ],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return root_df

# ---------------------------------------------------------
# 3) Aplicar para cada query
# ---------------------------------------------------------
root_candidates_by_query = {}

for _, row in workload_conceptual_df.iterrows():
    query_name = row["query_name"]
    terminals = row["entities_touched"]

    root_candidates_df = extract_root_candidates(terminals, conceptual_adj)
    root_candidates_by_query[query_name] = root_candidates_df

    print("\n" + "=" * 80)
    print(f"Candidatos iniciais de raiz para {query_name}")
    display(root_candidates_df)


Candidatos iniciais de raiz para Q1_Login


,candidate_root,avg_distance_to_other_touched_entities,max_distance_to_other_touched_entities,direct_links_to_other_touched_entities
0,User,0.0,0,0



Candidatos iniciais de raiz para Q2_SimpleSearch


,candidate_root,avg_distance_to_other_touched_entities,max_distance_to_other_touched_entities,direct_links_to_other_touched_entities
0,WatchItem,0.0,0,0



Candidatos iniciais de raiz para Q3_AddEntitiesAndRelationships


,candidate_root,avg_distance_to_other_touched_entities,max_distance_to_other_touched_entities,direct_links_to_other_touched_entities
0,Person,1.0,1,2
1,WatchItem,1.0,1,2
2,Role,1.0,1,2



Candidatos iniciais de raiz para Q4_Recommendation


,candidate_root,avg_distance_to_other_touched_entities,max_distance_to_other_touched_entities,direct_links_to_other_touched_entities
0,WatchItem,1.00,1,4
1,Series,1.50,2,2
2,Episode,1.50,2,2
3,Genre,1.75,2,1
4,Movie,1.75,2,1



Candidatos iniciais de raiz para Q5_AllPersonsOfTypeForWatchItem


,candidate_root,avg_distance_to_other_touched_entities,max_distance_to_other_touched_entities,direct_links_to_other_touched_entities
0,WatchItem,1.0,1,2
1,Person,1.0,1,2
2,Role,1.0,1,2


In [8]:
# =========================================================
# BLOCO V4 — DEFINIR A RAIZ ESCOLHIDA POR QUERY
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Definição manual/assistida das raízes escolhidas
# ---------------------------------------------------------
# Observação:
# aqui estamos registrando a decisão provisória da raiz
# para o experimento documental.
#
# A ideia é:
# - usar a heurística estrutural como apoio;
# - aplicar desempate semântico quando necessário.
selected_roots_df = pd.DataFrame([
    {
        "query_name": "Q1_Login",
        "selected_root": "User",
        "root_decision_rationale": (
            "A query toca apenas User; User é a raiz natural."
        )
    },
    {
        "query_name": "Q2_SimpleSearch",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "A query toca apenas WatchItem; WatchItem é a raiz natural."
        )
    },
    {
        "query_name": "Q3_AddEntitiesAndRelationships",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "Embora haja empate estrutural entre Person, Role e WatchItem, "
            "WatchItem foi escolhido como raiz documental provisória porque "
            "o vínculo criado é centrado no item e a modelagem documental do "
            "workload tende a se organizar ao redor de WatchItem."
        )
    },
    {
        "query_name": "Q4_Recommendation",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "WatchItem é claramente a entidade central da query e concentra "
            "Genre, Movie, Series e Episode no subgrafo conceitual."
        )
    },
    {
        "query_name": "Q5_AllPersonsOfTypeForWatchItem",
        "selected_root": "WatchItem",
        "root_decision_rationale": (
            "Embora haja empate estrutural entre Person, Role e WatchItem, "
            "a query é semanticamente centrada no watch item, o que favorece "
            "WatchItem como raiz documental."
        )
    },
])

print("Raízes documentais provisórias selecionadas:")
display(selected_roots_df)

Raízes documentais provisórias selecionadas:


,query_name,selected_root,root_decision_rationale
0,Q1_Login,User,A query toca apenas User; User é a raiz natural.
1,Q2_SimpleSearch,WatchItem,A query toca apenas WatchItem; WatchItem é a r...
2,Q3_AddEntitiesAndRelationships,WatchItem,"Embora haja empate estrutural entre Person, Ro..."
3,Q4_Recommendation,WatchItem,WatchItem é claramente a entidade central da q...
4,Q5_AllPersonsOfTypeForWatchItem,WatchItem,"Embora haja empate estrutural entre Person, Ro..."


In [9]:
# =========================================================
# BLOCO V5 — CALCULAR D(E, er, Qi)
# =========================================================

from collections import defaultdict, deque
import pandas as pd

# ---------------------------------------------------------
# 1) Índice auxiliar: mapear edge_id -> aresta conceitual
# ---------------------------------------------------------
edge_by_id = {
    edge["edge_id"]: edge
    for edge in conceptual_edges
}

# ---------------------------------------------------------
# 2) Função para montar adjacência do subgrafo da query
# ---------------------------------------------------------
def build_subgraph_adjacency_from_edge_ids(edge_ids):
    """
    Monta uma adjacência não-direcionada a partir da lista de edge_ids
    escolhida para representar o subgrafo conceitual mínimo da query.
    """
    adj = defaultdict(list)

    for edge_id in edge_ids:
        edge = edge_by_id[edge_id]
        src = edge["source"]
        tgt = edge["target"]

        adj[src].append(tgt)
        adj[tgt].append(src)

    return adj

# ---------------------------------------------------------
# 3) Função para calcular distâncias no subgrafo
# ---------------------------------------------------------
def shortest_distance_in_subgraph(root, target, sub_adj):
    """
    Calcula a menor distância entre root e target no subgrafo da query.
    """
    if root == target:
        return 0

    visited = set([root])
    queue = deque([(root, 0)])

    while queue:
        current, dist = queue.popleft()

        for neighbor in sub_adj[current]:
            if neighbor == target:
                return dist + 1

            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))

    return None

# ---------------------------------------------------------
# 4) Aplicar para cada query
# ---------------------------------------------------------
depth_rows = []

# junta Rc com as raízes escolhidas
rc_with_roots_df = rc_df.merge(selected_roots_df, on="query_name", how="left")

for _, row in rc_with_roots_df.iterrows():
    query_name = row["query_name"]
    entities_touched = row["entities_touched"]
    selected_root = row["selected_root"]
    selected_edge_ids = row["Rc_selected_edges"]

    sub_adj = build_subgraph_adjacency_from_edge_ids(selected_edge_ids)

    entity_distances = {}
    for entity in entities_touched:
        d = shortest_distance_in_subgraph(selected_root, entity, sub_adj)
        entity_distances[entity] = d

    # profundidade = maior distância da raiz até qualquer entidade tocada
    valid_distances = [d for d in entity_distances.values() if d is not None]
    depth_value = max(valid_distances) if valid_distances else 0

    depth_rows.append({
        "query_name": query_name,
        "selected_root": selected_root,
        "Rc": row["Rc"],
        "D_value": depth_value,
        "entity_distances_from_root": entity_distances,
        "Rc_selected_edges": selected_edge_ids,
        "root_decision_rationale": row["root_decision_rationale"]
    })

depth_df = pd.DataFrame(depth_rows)

print("Extração de D(E, er, Qi):")
display(depth_df)

Extração de D(E, er, Qi):


,query_name,selected_root,Rc,D_value,entity_distances_from_root,Rc_selected_edges,root_decision_rationale
0,Q1_Login,User,0,0,{'User': 0},[],A query toca apenas User; User é a raiz natural.
1,Q2_SimpleSearch,WatchItem,0,0,{'WatchItem': 0},[],A query toca apenas WatchItem; WatchItem é a r...
2,Q3_AddEntitiesAndRelationships,WatchItem,2,2,"{'Person': 2, 'WatchItem': 0, 'Role': 1}","[Person -- Role [acts_in_via_role], Role -- Wa...","Embora haja empate estrutural entre Person, Ro..."
3,Q4_Recommendation,WatchItem,4,1,"{'WatchItem': 0, 'Genre': 1, 'Movie': 1, 'Seri...","[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem é claramente a entidade central da q...
4,Q5_AllPersonsOfTypeForWatchItem,WatchItem,2,2,"{'WatchItem': 0, 'Person': 2, 'Role': 1}","[Person -- Role [acts_in_via_role], Role -- Wa...","Embora haja empate estrutural entre Person, Ro..."


In [10]:
# =========================================================
# BLOCO 1 — GERADOR DE DADOS SINTÉTICOS IMDb
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List
import pandas as pd

@dataclass
class ScenarioDataBundle:
    dataset_name: str
    tables: Dict[str, pd.DataFrame] = field(default_factory=dict)

    def table_names(self) -> List[str]:
        return list(self.tables.keys())

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "table_name": name,
                "rows": len(df),
                "columns": list(df.columns)
            }
            for name, df in self.tables.items()
        ]).sort_values("table_name").reset_index(drop=True)

In [11]:
# =========================================================
# BLOCO 2 — GERADOR DE DADOS SINTÉTICOS IMDb (CORRIGIDO)
# =========================================================

import random
import numpy as np
import pandas as pd

def generate_imdb_synthetic_data(
    n_users: int = 100,
    n_persons: int = 120,
    n_watchitems: int = 80,
    n_genres: int = 10,
    seed: int = 42
) -> "ScenarioDataBundle":
    """
    Gera um dataset sintético simplificado para o cenário IMDb.
    """

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    # 1) USERS
    users = pd.DataFrame([
        {
            "user_id": i,
            "username": f"user_{i}",
            "password": f"pass_{i}",
            "email": f"user_{i}@mail.com",
            "last_login": f"2026-01-{(i % 28) + 1:02d}"
        }
        for i in range(1, n_users + 1)
    ])

    # 2) PERSONS
    persons = pd.DataFrame([
        {
            "person_id": i,
            "name": f"Person {i}",
            "date_of_birth": f"{1970 + (i % 30)}-{(i % 12) + 1:02d}-{(i % 28) + 1:02d}",
            "gender": "M" if i % 2 == 0 else "F"
        }
        for i in range(1, n_persons + 1)
    ])

    # 3) GENRES
    genre_names = [
        "Action", "Drama", "Comedy", "Thriller", "SciFi",
        "Fantasy", "Romance", "Crime", "Adventure", "Mystery"
    ][:n_genres]

    genres = pd.DataFrame([
        {"genre_id": i + 1, "name": genre_names[i]}
        for i in range(len(genre_names))
    ])

    # 4) WATCHITEMS
    item_types = np_rng.choice(
        ["Movie", "Series", "Episode"],
        size=n_watchitems,
        p=[0.45, 0.20, 0.35]
    )

    watchitems_rows = []
    for i in range(1, n_watchitems + 1):
        genre_id = int(np_rng.integers(1, len(genres) + 1))
        release_year = int(np_rng.integers(1990, 2026))

        watchitems_rows.append({
            "watchitem_id": i,
            "title": f"Title {i}",
            "release_year": release_year,
            "avg_rating": round(float(np_rng.uniform(1.0, 10.0)), 2),
            "genre_id": genre_id,
            "item_type": item_types[i - 1]
        })

    watchitems = pd.DataFrame(watchitems_rows)

    # 5) MOVIES
    movies = watchitems[watchitems["item_type"] == "Movie"][["watchitem_id"]].copy()
    movies["length_min"] = np_rng.integers(80, 181, size=len(movies))
    movies["media"] = np_rng.choice(["Cinema", "Streaming", "TV"], size=len(movies))
    movies["income"] = np_rng.integers(100000, 1000000000, size=len(movies))

    # 6) SERIES
    series = watchitems[watchitems["item_type"] == "Series"][["watchitem_id"]].copy()
    series["seasons"] = np_rng.integers(1, 8, size=len(series))
    series["network"] = np_rng.choice(["HBO", "Netflix", "Prime", "Disney"], size=len(series))

    # 7) EPISODES
    episodes = watchitems[watchitems["item_type"] == "Episode"][["watchitem_id"]].copy()
    episodes["season"] = np_rng.integers(1, 8, size=len(episodes))
    episodes["episode_number"] = np_rng.integers(1, 25, size=len(episodes))
    episodes["length_min"] = np_rng.integers(20, 61, size=len(episodes))

    if len(series) > 0 and len(episodes) > 0:
        series_ids = series["watchitem_id"].tolist()
        episodes["series_watchitem_id"] = [rng.choice(series_ids) for _ in range(len(episodes))]
    else:
        episodes["series_watchitem_id"] = None

    # 8) ROLES
    role_types = ["Actor", "Director", "Producer"]

    roles_rows = []
    role_id = 1
    for watchitem_id in watchitems["watchitem_id"]:
        n_links = int(np_rng.integers(2, 6))
        chosen_persons = rng.sample(persons["person_id"].tolist(), k=min(n_links, len(persons)))

        for person_id in chosen_persons:
            roles_rows.append({
                "role_id": role_id,
                "person_id": person_id,
                "watchitem_id": watchitem_id,
                "role_type": rng.choice(role_types)
            })
            role_id += 1

    roles = pd.DataFrame(roles_rows)

    # 9) RATES
    rates_rows = []
    rate_id = 1

    for user_id in users["user_id"]:
        n_user_ratings = int(np_rng.integers(2, 10))
        chosen_items = rng.sample(
            watchitems["watchitem_id"].tolist(),
            k=min(n_user_ratings, len(watchitems))
        )

        for watchitem_id in chosen_items:
            rating_value = int(np_rng.integers(1, 11))
            verbal = (
                "Excellent" if rating_value >= 9 else
                "Good" if rating_value >= 7 else
                "Average" if rating_value >= 5 else
                "Bad"
            )

            rates_rows.append({
                "rate_id": rate_id,
                "user_id": user_id,
                "watchitem_id": watchitem_id,
                "rating": rating_value,
                "verbal_rating": verbal
            })
            rate_id += 1

    rates = pd.DataFrame(rates_rows)

    return ScenarioDataBundle(
        dataset_name="IMDb",
        tables={
            "users": users,
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
        }
    )

In [12]:
# =========================================================
# BLOCO 3 — GERAR O DATASET IMDb SINTÉTICO
# =========================================================

import pandas as pd

# Checagem simples para evitar erro confuso se o bloco anterior não tiver rodado
if "generate_imdb_synthetic_data" not in globals():
    raise NameError(
        "A função 'generate_imdb_synthetic_data' não está definida. "
        "Rode primeiro o bloco anterior que cria essa função."
    )

# Gera o dataset sintético
imdb_data_bundle = generate_imdb_synthetic_data(
    n_users=100,
    n_persons=120,
    n_watchitems=80,
    n_genres=10,
    seed=42
)

# Mostra um resumo das tabelas geradas
print("Resumo das tabelas geradas:")
display(imdb_data_bundle.summary())

# Mostra algumas amostras para inspeção
print("Prévia: users")
display(imdb_data_bundle.tables["users"].head())

print("Prévia: watchitems")
display(imdb_data_bundle.tables["watchitems"].head())

print("Prévia: roles")
display(imdb_data_bundle.tables["roles"].head())

print("Prévia: rates")
display(imdb_data_bundle.tables["rates"].head())

Resumo das tabelas geradas:


,table_name,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,users,100,"[user_id, username, password, email, last_login]"
8,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."


Prévia: users


,user_id,username,password,email,last_login
0,1,user_1,pass_1,user_1@mail.com,2026-01-02
1,2,user_2,pass_2,user_2@mail.com,2026-01-03
2,3,user_3,pass_3,user_3@mail.com,2026-01-04
3,4,user_4,pass_4,user_4@mail.com,2026-01-05
4,5,user_5,pass_5,user_5@mail.com,2026-01-06


Prévia: watchitems


,watchitem_id,title,release_year,avg_rating,genre_id,item_type
0,1,Title 1,2013,4.66,10,Episode
1,2,Title 2,2019,2.50,5,Movie
2,3,Title 3,1990,1.81,4,Episode
3,4,Title 4,2016,5.16,8,Episode
4,5,Title 5,1995,5.51,8,Movie


Prévia: roles


,role_id,person_id,watchitem_id,role_type
0,1,54,1,Producer
1,2,29,1,Director
2,3,58,1,Actor
3,4,98,2,Producer
4,5,104,2,Director


Prévia: rates


,rate_id,user_id,watchitem_id,rating,verbal_rating
0,1,1,26,1,Bad
1,2,1,47,1,Bad
2,3,1,56,5,Average
3,4,1,9,8,Good
4,5,1,43,9,Excellent


In [13]:
# =========================================================
# BLOCO V6 — EXTRair CARDINALIDADE OBSERVADA NO DATASET
# =========================================================
# TO EXTRAXT CARDINALITY

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
required_names = [
    "imdb_data_bundle"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos do dataset IMDb."
        )

# ---------------------------------------------------------
# 2) Recuperar tabelas do bundle
# ---------------------------------------------------------
users_df = imdb_data_bundle.tables["users"]
persons_df = imdb_data_bundle.tables["persons"]
genres_df = imdb_data_bundle.tables["genres"]
watchitems_df = imdb_data_bundle.tables["watchitems"]
movies_df = imdb_data_bundle.tables["movies"]
series_df = imdb_data_bundle.tables["series"]
episodes_df = imdb_data_bundle.tables["episodes"]
roles_df = imdb_data_bundle.tables["roles"]
rates_df = imdb_data_bundle.tables["rates"]

# ---------------------------------------------------------
# 3) Função auxiliar para resumir cardinalidade observada
# ---------------------------------------------------------
def summarize_observed_cardinality(
    df: pd.DataFrame,
    left_col: str,
    right_col: str,
    relationship_name: str
) -> dict:
    """
    Resume a cardinalidade observada entre duas colunas de ligação.

    left_col  = lado esquerdo da relação
    right_col = lado direito da relação

    O objetivo é medir:
    - quantos registros do lado direito aparecem por valor do lado esquerdo
    - quantos registros do lado esquerdo aparecem por valor do lado direito
    """
    left_to_right = df.groupby(left_col)[right_col].nunique()
    right_to_left = df.groupby(right_col)[left_col].nunique()

    return {
        "relationship_name": relationship_name,
        "left_entity_key": left_col,
        "right_entity_key": right_col,
        "left_distinct_values": df[left_col].nunique(),
        "right_distinct_values": df[right_col].nunique(),
        "avg_right_per_left": round(left_to_right.mean(), 2),
        "max_right_per_left": int(left_to_right.max()),
        "avg_left_per_right": round(right_to_left.mean(), 2),
        "max_left_per_right": int(right_to_left.max()),
    }

# ---------------------------------------------------------
# 4) Extrair cardinalidades principais do IMDb
# ---------------------------------------------------------
cardinality_rows = []

# Genre -> WatchItem
cardinality_rows.append(
    summarize_observed_cardinality(
        df=watchitems_df,
        left_col="genre_id",
        right_col="watchitem_id",
        relationship_name="Genre -- WatchItem"
    )
)

# Person -> WatchItem via Role
cardinality_rows.append(
    summarize_observed_cardinality(
        df=roles_df,
        left_col="person_id",
        right_col="watchitem_id",
        relationship_name="Person -- WatchItem (via Role)"
    )
)

# WatchItem -> Person via Role
cardinality_rows.append(
    summarize_observed_cardinality(
        df=roles_df,
        left_col="watchitem_id",
        right_col="person_id",
        relationship_name="WatchItem -- Person (via Role)"
    )
)

# User -> WatchItem via Rate
cardinality_rows.append(
    summarize_observed_cardinality(
        df=rates_df,
        left_col="user_id",
        right_col="watchitem_id",
        relationship_name="User -- WatchItem (via Rate)"
    )
)

# Series -> Episode
episodes_with_series = episodes_df.dropna(subset=["series_watchitem_id"]).copy()

if not episodes_with_series.empty:
    cardinality_rows.append(
        summarize_observed_cardinality(
            df=episodes_with_series,
            left_col="series_watchitem_id",
            right_col="watchitem_id",
            relationship_name="Series -- Episode"
        )
    )

cardinality_df = pd.DataFrame(cardinality_rows)

print("Cardinalidade observada no dataset sintético:")
display(cardinality_df)

Cardinalidade observada no dataset sintético:


,relationship_name,left_entity_key,right_entity_key,left_distinct_values,right_distinct_values,avg_right_per_left,max_right_per_left,avg_left_per_right,max_left_per_right
0,Genre -- WatchItem,genre_id,watchitem_id,10,80,8.00,12,1.00,1
1,Person -- WatchItem (via Role),person_id,watchitem_id,114,80,2.54,9,3.62,5
2,WatchItem -- Person (via Role),watchitem_id,person_id,80,114,3.62,5,2.54,9
3,User -- WatchItem (via Rate),user_id,watchitem_id,100,80,5.47,9,6.84,13
4,Series -- Episode,series_watchitem_id,watchitem_id,11,31,2.82,5,1.00,1


In [14]:
# =========================================================
# BLOCO V7 — CLASSIFICAÇÃO SEMÂNTICA DOS RELACIONAMENTOS
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "conceptual_relationships_df" not in globals():
    raise NameError(
        "O DataFrame 'conceptual_relationships_df' não está definido. "
        "Rode primeiro o bloco que extrai os relacionamentos conceituais."
    )

# ---------------------------------------------------------
# 2) Classificação semântica manual-assistida
# ---------------------------------------------------------
# Observação importante:
# esta classificação não vem diretamente dos dados,
# mas sim da interpretação do esquema conceitual.
#
# Tipos usados:
# - containment : uma entidade faz parte naturalmente da outra
# - association : duas entidades independentes estão ligadas
# - associative : a ligação é mediada por uma entidade/semântica associativa
relationship_semantics_df = pd.DataFrame([
    {
        "source": "Person",
        "target": "Role",
        "relationship_name": "acts_in_via_role",
        "semantic_type": "associative",
        "rationale": (
            "Role funciona como entidade associativa entre Person e WatchItem, "
            "carregando a semântica do papel desempenhado."
        )
    },
    {
        "source": "Role",
        "target": "WatchItem",
        "relationship_name": "role_of_watchitem",
        "semantic_type": "associative",
        "rationale": (
            "Role conecta semanticamente uma pessoa a um item assistível, "
            "representando uma associação qualificada."
        )
    },
    {
        "source": "Person",
        "target": "WatchItem",
        "relationship_name": "directs",
        "semantic_type": "association",
        "rationale": (
            "Person e WatchItem permanecem entidades independentes; "
            "a relação expressa associação, não contenção."
        )
    },
    {
        "source": "Person",
        "target": "WatchItem",
        "relationship_name": "produces",
        "semantic_type": "association",
        "rationale": (
            "Person e WatchItem permanecem entidades independentes; "
            "a relação expressa associação, não contenção."
        )
    },
    {
        "source": "Genre",
        "target": "WatchItem",
        "relationship_name": "has_genre",
        "semantic_type": "association",
        "rationale": (
            "Genre é um descritor classificatório do item. Pode ser embutido "
            "por conveniência, mas semanticamente continua sendo associação."
        )
    },
    {
        "source": "User",
        "target": "Rate",
        "relationship_name": "rates",
        "semantic_type": "associative",
        "rationale": (
            "Rate é uma entidade associativa entre User e WatchItem, "
            "carregando o valor da avaliação."
        )
    },
    {
        "source": "Rate",
        "target": "WatchItem",
        "relationship_name": "rate_of_watchitem",
        "semantic_type": "associative",
        "rationale": (
            "Rate conecta User e WatchItem e contém semântica própria da avaliação."
        )
    },
    {
        "source": "Series",
        "target": "WatchItem",
        "relationship_name": "series_is_watchitem",
        "semantic_type": "association",
        "rationale": (
            "Aqui a relação representa mais uma especialização/supertipo "
            "do que contenção propriamente dita."
        )
    },
    {
        "source": "Movie",
        "target": "WatchItem",
        "relationship_name": "movie_is_watchitem",
        "semantic_type": "association",
        "rationale": (
            "Aqui a relação representa mais uma especialização/supertipo "
            "do que contenção propriamente dita."
        )
    },
    {
        "source": "Episode",
        "target": "WatchItem",
        "relationship_name": "episode_is_watchitem",
        "semantic_type": "association",
        "rationale": (
            "Aqui a relação representa mais uma especialização/supertipo "
            "do que contenção propriamente dita."
        )
    },
    {
        "source": "Series",
        "target": "Episode",
        "relationship_name": "series_contains_episode",
        "semantic_type": "containment",
        "rationale": (
            "Episode faz parte naturalmente de Series; este é o caso mais claro "
            "de containment no esquema IMDb."
        )
    },
])

print("Classificação semântica dos relacionamentos do IMDb:")
display(relationship_semantics_df)

Classificação semântica dos relacionamentos do IMDb:


,source,target,relationship_name,semantic_type,rationale
0,Person,Role,acts_in_via_role,associative,Role funciona como entidade associativa entre ...
1,Role,WatchItem,role_of_watchitem,associative,Role conecta semanticamente uma pessoa a um it...
2,Person,WatchItem,directs,association,Person e WatchItem permanecem entidades indepe...
3,Person,WatchItem,produces,association,Person e WatchItem permanecem entidades indepe...
4,Genre,WatchItem,has_genre,association,Genre é um descritor classificatório do item. ...
5,User,Rate,rates,associative,Rate é uma entidade associativa entre User e W...
6,Rate,WatchItem,rate_of_watchitem,associative,Rate conecta User e WatchItem e contém semânti...
7,Series,WatchItem,series_is_watchitem,association,Aqui a relação representa mais uma especializa...
8,Movie,WatchItem,movie_is_watchitem,association,Aqui a relação representa mais uma especializa...
9,Episode,WatchItem,episode_is_watchitem,association,Aqui a relação representa mais uma especializa...


In [15]:
# =========================================================
# BLOCO V8 — TIPOS DE RELACIONAMENTO TOCADOS POR CADA QUERY
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "rc_df",
    "relationship_semantics_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Criar edge_id compatível com o usado em Rc_selected_edges
# ---------------------------------------------------------
relationship_semantics_df = relationship_semantics_df.copy()
relationship_semantics_df["edge_id"] = relationship_semantics_df.apply(
    lambda row: f'{row["source"]} -- {row["target"]} [{row["relationship_name"]}]',
    axis=1
)

# ---------------------------------------------------------
# 3) Expandir os relacionamentos tocados por cada query
# ---------------------------------------------------------
query_relationship_rows = []

for _, row in rc_df.iterrows():
    query_name = row["query_name"]
    selected_edges = row["Rc_selected_edges"]

    for edge_id in selected_edges:
        match = relationship_semantics_df[
            relationship_semantics_df["edge_id"] == edge_id
        ]

        if len(match) == 0:
            query_relationship_rows.append({
                "query_name": query_name,
                "edge_id": edge_id,
                "semantic_type": None,
                "rationale": "Relacionamento não classificado"
            })
        else:
            for _, match_row in match.iterrows():
                query_relationship_rows.append({
                    "query_name": query_name,
                    "edge_id": edge_id,
                    "semantic_type": match_row["semantic_type"],
                    "rationale": match_row["rationale"]
                })

query_relationship_semantics_df = pd.DataFrame(query_relationship_rows)

print("Tipos semânticos dos relacionamentos tocados por cada query:")
display(query_relationship_semantics_df)

# ---------------------------------------------------------
# 4) Resumo por query
# ---------------------------------------------------------
if not query_relationship_semantics_df.empty:
    semantic_summary_df = (
        query_relationship_semantics_df
        .groupby(["query_name", "semantic_type"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["query_name", "semantic_type"])
        .reset_index(drop=True)
    )
else:
    semantic_summary_df = pd.DataFrame()

print("Resumo semântico por query:")
display(semantic_summary_df)

Tipos semânticos dos relacionamentos tocados por cada query:


,query_name,edge_id,semantic_type,rationale
0,Q3_AddEntitiesAndRelationships,Person -- Role [acts_in_via_role],associative,Role funciona como entidade associativa entre ...
1,Q3_AddEntitiesAndRelationships,Role -- WatchItem [role_of_watchitem],associative,Role conecta semanticamente uma pessoa a um it...
2,Q4_Recommendation,Genre -- WatchItem [has_genre],association,Genre é um descritor classificatório do item. ...
3,Q4_Recommendation,Series -- WatchItem [series_is_watchitem],association,Aqui a relação representa mais uma especializa...
4,Q4_Recommendation,Movie -- WatchItem [movie_is_watchitem],association,Aqui a relação representa mais uma especializa...
5,Q4_Recommendation,Episode -- WatchItem [episode_is_watchitem],association,Aqui a relação representa mais uma especializa...
6,Q5_AllPersonsOfTypeForWatchItem,Person -- Role [acts_in_via_role],associative,Role funciona como entidade associativa entre ...
7,Q5_AllPersonsOfTypeForWatchItem,Role -- WatchItem [role_of_watchitem],associative,Role conecta semanticamente uma pessoa a um it...


Resumo semântico por query:


,query_name,semantic_type,count
0,Q3_AddEntitiesAndRelationships,associative,2
1,Q4_Recommendation,association,4
2,Q5_AllPersonsOfTypeForWatchItem,associative,2


In [16]:
# =========================================================
# BLOCO V9 — ANÁLISE ESTRUTURAL DAS ARESTAS POR PROFUNDIDADE
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "depth_df",
    "edge_by_id"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função auxiliar:
# verifica se uma aresta pode ser absorvida pelo documento
# em uma profundidade de embutimento 'd'
# ---------------------------------------------------------
def edge_is_absorbed_at_depth(edge_id, entity_distances_from_root, d):
    """
    Uma aresta é considerada absorvida se as duas entidades
    de suas extremidades estiverem cobertas pela profundidade d.
    """
    edge = edge_by_id[edge_id]
    src = edge["source"]
    tgt = edge["target"]

    dist_src = entity_distances_from_root.get(src, None)
    dist_tgt = entity_distances_from_root.get(tgt, None)

    # Se alguma entidade não estiver no subgrafo da query,
    # tratamos como não absorvida.
    if dist_src is None or dist_tgt is None:
        return False

    return (dist_src <= d) and (dist_tgt <= d)

# ---------------------------------------------------------
# 3) Expandir aresta por aresta, profundidade por profundidade
# ---------------------------------------------------------
edge_depth_rows = []

for _, row in depth_df.iterrows():
    query_name = row["query_name"]
    selected_root = row["selected_root"]
    rc_value = int(row["Rc"])
    max_depth_for_query = int(row["D_value"])
    entity_distances = row["entity_distances_from_root"]
    selected_edge_ids = row["Rc_selected_edges"]

    # Para cada profundidade possível: 0 até D_value
    for d in range(0, max_depth_for_query + 1):
        for edge_id in selected_edge_ids:
            edge = edge_by_id[edge_id]

            absorbed = edge_is_absorbed_at_depth(
                edge_id=edge_id,
                entity_distances_from_root=entity_distances,
                d=d
            )

            edge_depth_rows.append({
                "query_name": query_name,
                "selected_root": selected_root,
                "test_depth": d,
                "Rc": rc_value,
                "edge_id": edge_id,
                "edge_source": edge["source"],
                "edge_target": edge["target"],
                "source_distance": entity_distances.get(edge["source"], None),
                "target_distance": entity_distances.get(edge["target"], None),
                "absorbed_by_document": absorbed
            })

edge_depth_df = pd.DataFrame(edge_depth_rows)

print("Análise estrutural das arestas por profundidade:")
display(edge_depth_df)

Análise estrutural das arestas por profundidade:


,query_name,selected_root,test_depth,Rc,edge_id,edge_source,edge_target,source_distance,target_distance,absorbed_by_document
0,Q3_AddEntitiesAndRelationships,WatchItem,0,2,Person -- Role [acts_in_via_role],Person,Role,2,1,False
1,Q3_AddEntitiesAndRelationships,WatchItem,0,2,Role -- WatchItem [role_of_watchitem],Role,WatchItem,1,0,False
2,Q3_AddEntitiesAndRelationships,WatchItem,1,2,Person -- Role [acts_in_via_role],Person,Role,2,1,False
3,Q3_AddEntitiesAndRelationships,WatchItem,1,2,Role -- WatchItem [role_of_watchitem],Role,WatchItem,1,0,True
4,Q3_AddEntitiesAndRelationships,WatchItem,2,2,Person -- Role [acts_in_via_role],Person,Role,2,1,True
5,Q3_AddEntitiesAndRelationships,WatchItem,2,2,Role -- WatchItem [role_of_watchitem],Role,WatchItem,1,0,True
6,Q4_Recommendation,WatchItem,0,4,Genre -- WatchItem [has_genre],Genre,WatchItem,1,0,False
7,Q4_Recommendation,WatchItem,0,4,Series -- WatchItem [series_is_watchitem],Series,WatchItem,1,0,False
8,Q4_Recommendation,WatchItem,0,4,Movie -- WatchItem [movie_is_watchitem],Movie,WatchItem,1,0,False
9,Q4_Recommendation,WatchItem,0,4,Episode -- WatchItem [episode_is_watchitem],Episode,WatchItem,1,0,False


In [17]:
# =========================================================
# BLOCO V10 — CALCULAR Re(Qi, er, D) E DeltaR
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "depth_df",
    "edge_depth_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Casos com Rc > 0
# Aqui resumimos as queries que realmente têm arestas conceituais
# ---------------------------------------------------------
if not edge_depth_df.empty:
    re_summary_df = (
        edge_depth_df
        .groupby(["query_name", "selected_root", "test_depth", "Rc"], as_index=False)
        .agg(
            n_edges_total=("edge_id", "count"),
            n_edges_absorbed=("absorbed_by_document", "sum")
        )
    )

    # Re = arestas que continuam explícitas em execução
    re_summary_df["Re"] = re_summary_df["n_edges_total"] - re_summary_df["n_edges_absorbed"]

    # DeltaR = redução obtida pela estrutura documental
    re_summary_df["DeltaR"] = re_summary_df["Rc"] - re_summary_df["Re"]

else:
    re_summary_df = pd.DataFrame(columns=[
        "query_name", "selected_root", "test_depth", "Rc",
        "n_edges_total", "n_edges_absorbed", "Re", "DeltaR"
    ])

# ---------------------------------------------------------
# 3) Casos triviais com Rc = 0
# Para Q1 e Q2, não há relacionamentos para absorver
# ---------------------------------------------------------
zero_rc_rows = []

for _, row in depth_df.iterrows():
    if int(row["Rc"]) == 0:
        zero_rc_rows.append({
            "query_name": row["query_name"],
            "selected_root": row["selected_root"],
            "test_depth": 0,
            "Rc": 0,
            "n_edges_total": 0,
            "n_edges_absorbed": 0,
            "Re": 0,
            "DeltaR": 0
        })

zero_rc_df = pd.DataFrame(zero_rc_rows)

# ---------------------------------------------------------
# 4) Unir tudo
# ---------------------------------------------------------
re_full_df = pd.concat([re_summary_df, zero_rc_df], ignore_index=True)

re_full_df = re_full_df.sort_values(
    by=["query_name", "test_depth"]
).reset_index(drop=True)

print("Resumo de Re(Qi, er, D) e DeltaR por profundidade:")
display(re_full_df)

Resumo de Re(Qi, er, D) e DeltaR por profundidade:


,query_name,selected_root,test_depth,Rc,n_edges_total,n_edges_absorbed,Re,DeltaR
0,Q1_Login,User,0,0,0,0,0,0
1,Q2_SimpleSearch,WatchItem,0,0,0,0,0,0
2,Q3_AddEntitiesAndRelationships,WatchItem,0,2,2,0,2,0
3,Q3_AddEntitiesAndRelationships,WatchItem,1,2,2,1,1,1
4,Q3_AddEntitiesAndRelationships,WatchItem,2,2,2,2,0,2
5,Q4_Recommendation,WatchItem,0,4,4,0,4,0
6,Q4_Recommendation,WatchItem,1,4,4,4,0,4
7,Q5_AllPersonsOfTypeForWatchItem,WatchItem,0,2,2,0,2,0
8,Q5_AllPersonsOfTypeForWatchItem,WatchItem,1,2,2,1,1,1
9,Q5_AllPersonsOfTypeForWatchItem,WatchItem,2,2,2,2,0,2


In [18]:
# =========================================================
# BLOCO V11 — PREPARAR O RESUMO SEMÂNTICO POR QUERY
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "workload_conceptual_df",
    "semantic_summary_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Pivotar o resumo semântico
# A ideia é transformar:
#   query_name | semantic_type | count
# em:
#   query_name | n_association | n_associative | n_containment
# ---------------------------------------------------------
if semantic_summary_df.empty:
    semantic_pivot_df = pd.DataFrame({
        "query_name": workload_conceptual_df["query_name"].tolist(),
        "n_association": 0,
        "n_associative": 0,
        "n_containment": 0
    })
else:
    semantic_pivot_df = (
        semantic_summary_df
        .pivot(index="query_name", columns="semantic_type", values="count")
        .fillna(0)
        .reset_index()
    )

    # Garante que todas as colunas existam
    for col in ["association", "associative", "containment"]:
        if col not in semantic_pivot_df.columns:
            semantic_pivot_df[col] = 0

    semantic_pivot_df = semantic_pivot_df.rename(columns={
        "association": "n_association",
        "associative": "n_associative",
        "containment": "n_containment"
    })

# ---------------------------------------------------------
# 3) Garantir que todas as queries do workload apareçam
# mesmo que alguma não tenha relacionamentos tocados
# ---------------------------------------------------------
semantic_pivot_df = (
    workload_conceptual_df[["query_name"]]
    .merge(semantic_pivot_df, on="query_name", how="left")
    .fillna(0)
)

# Converte para inteiro
for col in ["n_association", "n_associative", "n_containment"]:
    semantic_pivot_df[col] = semantic_pivot_df[col].astype(int)

print("Resumo semântico por query em formato consolidado:")
display(semantic_pivot_df)

Resumo semântico por query em formato consolidado:


,query_name,n_association,n_associative,n_containment
0,Q1_Login,0,0,0
1,Q2_SimpleSearch,0,0,0
2,Q3_AddEntitiesAndRelationships,0,2,0
3,Q4_Recommendation,4,0,0
4,Q5_AllPersonsOfTypeForWatchItem,0,2,0


In [19]:
# =========================================================
# BLOCO V12 — MATRIZ ANALÍTICA DO IMDb (EXPANDIDA POR PROFUNDIDADE)
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "workload_conceptual_df",
    "rc_df",
    "selected_roots_df",
    "depth_df",
    "re_full_df",
    "semantic_pivot_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Selecionar e renomear colunas-base
# ---------------------------------------------------------
workload_base_df = workload_conceptual_df.copy()

# Renomeia para deixar a matriz mais autoexplicativa
workload_base_df = workload_base_df.rename(columns={
    "entities_touched": "Qi_entities_touched",
    "n_entities_touched": "Qi_n_entities_touched"
})

# ---------------------------------------------------------
# 3) Base estrutural por query
# ---------------------------------------------------------
query_structure_df = (
    rc_df[["query_name", "Rc", "Rc_selected_edges"]]
    .merge(
        selected_roots_df[["query_name", "selected_root", "root_decision_rationale"]],
        on="query_name",
        how="left"
    )
    .merge(
        depth_df[["query_name", "D_value", "entity_distances_from_root"]],
        on="query_name",
        how="left"
    )
    .merge(
        semantic_pivot_df,
        on="query_name",
        how="left"
    )
)

# ---------------------------------------------------------
# 4) Unir com os resultados por profundidade
# Agora cada linha representa:
# - uma query
# - uma profundidade testada
# ---------------------------------------------------------
analytical_matrix_df = (
    workload_base_df
    .merge(query_structure_df, on="query_name", how="left")
    .merge(
        re_full_df[["query_name", "test_depth", "Re", "DeltaR"]],
        on="query_name",
        how="left"
    )
)

# ---------------------------------------------------------
# 5) Criar algumas colunas auxiliares de interpretação
# ---------------------------------------------------------

# Proporção de redução estrutural:
# DeltaR / Rc
def safe_ratio(delta_r, rc):
    if rc is None or rc == 0:
        return np.nan
    return delta_r / rc

analytical_matrix_df["DeltaR_ratio"] = analytical_matrix_df.apply(
    lambda row: safe_ratio(row["DeltaR"], row["Rc"]),
    axis=1
)

# Label simples da cobertura estrutural
def classify_structural_coverage(row):
    rc = row["Rc"]
    re = row["Re"]

    if rc == 0:
        return "No structural traversal"
    if re == 0:
        return "Full absorption"
    if re < rc:
        return "Partial absorption"
    return "No absorption"

analytical_matrix_df["document_structural_coverage"] = analytical_matrix_df.apply(
    classify_structural_coverage,
    axis=1
)

# ---------------------------------------------------------
# 6) Ordenar para leitura
# ---------------------------------------------------------
analytical_matrix_df = analytical_matrix_df.sort_values(
    by=["query_name", "test_depth"]
).reset_index(drop=True)

print("Matriz analítica consolidada do IMDb (expandida por profundidade):")
display(analytical_matrix_df)

Matriz analítica consolidada do IMDb (expandida por profundidade):


,query_name,query_type,Qi_entities_touched,abstract_query,Qi_n_entities_touched,Rc,Rc_selected_edges,selected_root,root_decision_rationale,D_value,entity_distances_from_root,n_association,n_associative,n_containment,test_depth,Re,DeltaR,DeltaR_ratio,document_structural_coverage
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1,0,[],User,A query toca apenas User; User é a raiz natural.,0,{'User': 0},0,0,0,0,0,0,NaN,No structural traversal
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1,0,[],WatchItem,A query toca apenas WatchItem; WatchItem é a r...,0,{'WatchItem': 0},0,0,0,0,0,0,NaN,No structural traversal
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'Person': 2, 'WatchItem': 0, 'Role': 1}",0,2,0,0,2,0,0.0,No absorption
3,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'Person': 2, 'WatchItem': 0, 'Role': 1}",0,2,0,1,1,1,0.5,Partial absorption
4,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'Person': 2, 'WatchItem': 0, 'Role': 1}",0,2,0,2,0,2,1.0,Full absorption
5,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,4,"[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem,WatchItem é claramente a entidade central da q...,1,"{'WatchItem': 0, 'Genre': 1, 'Movie': 1, 'Seri...",4,0,0,0,4,0,0.0,No absorption
6,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,4,"[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem,WatchItem é claramente a entidade central da q...,1,"{'WatchItem': 0, 'Genre': 1, 'Movie': 1, 'Seri...",4,0,0,1,0,4,1.0,Full absorption
7,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'WatchItem': 0, 'Person': 2, 'Role': 1}",0,2,0,0,2,0,0.0,No absorption
8,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'WatchItem': 0, 'Person': 2, 'Role': 1}",0,2,0,1,1,1,0.5,Partial absorption
9,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'WatchItem': 0, 'Person': 2, 'Role': 1}",0,2,0,2,0,2,1.0,Full absorption


In [20]:
# =========================================================
# BLOCO V13 — MATRIZ ANALÍTICA COMPACTA POR QUERY
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função para escolher a linha representativa da query
# Regra:
# - se houver profundidade com absorção total, pegar a menor delas
# - senão, pegar a maior profundidade testada
# ---------------------------------------------------------
def select_representative_row(group: pd.DataFrame) -> pd.Series:
    full_absorption = group[group["document_structural_coverage"] == "Full absorption"]

    if not full_absorption.empty:
        # menor profundidade que já resolve tudo
        selected = full_absorption.sort_values("test_depth").iloc[0]
    else:
        # caso raro: pega a maior profundidade testada
        selected = group.sort_values("test_depth", ascending=False).iloc[0]

    return selected

# ---------------------------------------------------------
# 3) Aplicar por query
# ---------------------------------------------------------
compact_rows = []

for query_name, group in analytical_matrix_df.groupby("query_name"):
    selected = select_representative_row(group)
    compact_rows.append(selected)

analytical_matrix_compact_df = pd.DataFrame(compact_rows).reset_index(drop=True)

# ---------------------------------------------------------
# 4) Renomear algumas colunas para deixar mais claro
# ---------------------------------------------------------
analytical_matrix_compact_df = analytical_matrix_compact_df.rename(columns={
    "test_depth": "selected_document_depth",
    "Re": "selected_Re",
    "DeltaR": "selected_DeltaR",
    "DeltaR_ratio": "selected_DeltaR_ratio",
    "document_structural_coverage": "selected_structural_coverage"
})

# ---------------------------------------------------------
# 5) Ordenar
# ---------------------------------------------------------
analytical_matrix_compact_df = analytical_matrix_compact_df.sort_values(
    by="query_name"
).reset_index(drop=True)

print("Matriz analítica compacta do IMDb (uma linha por query):")
display(analytical_matrix_compact_df)

Matriz analítica compacta do IMDb (uma linha por query):


,query_name,query_type,Qi_entities_touched,abstract_query,Qi_n_entities_touched,Rc,Rc_selected_edges,selected_root,root_decision_rationale,D_value,entity_distances_from_root,n_association,n_associative,n_containment,selected_document_depth,selected_Re,selected_DeltaR,selected_DeltaR_ratio,selected_structural_coverage
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1,0,[],User,A query toca apenas User; User é a raiz natural.,0,{'User': 0},0,0,0,0,0,0,NaN,No structural traversal
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1,0,[],WatchItem,A query toca apenas WatchItem; WatchItem é a r...,0,{'WatchItem': 0},0,0,0,0,0,0,NaN,No structural traversal
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'Person': 2, 'WatchItem': 0, 'Role': 1}",0,2,0,2,0,2,1.0,Full absorption
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,4,"[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem,WatchItem é claramente a entidade central da q...,1,"{'WatchItem': 0, 'Genre': 1, 'Movie': 1, 'Seri...",4,0,0,1,0,4,1.0,Full absorption
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,"{'WatchItem': 0, 'Person': 2, 'Role': 1}",0,2,0,2,0,2,1.0,Full absorption


In [21]:
# =========================================================
# BLOCO V14 — ENTIDADES EXPLICITAMENTE MODIFICADAS POR QUERY
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "workload_conceptual_df" not in globals():
    raise NameError(
        "O DataFrame 'workload_conceptual_df' não está definido. "
        "Rode primeiro os blocos do workload conceitual."
    )

# ---------------------------------------------------------
# 2) Definição manual-assistida das entidades modificadas
# Regra adotada aqui:
# - só entra a entidade que é explicitamente inserida/atualizada/removida
# - entidades apenas referenciadas pela query ficam de fora
# ---------------------------------------------------------
query_update_map_df = pd.DataFrame([
    {
        "query_name": "Q1_Login",
        "updated_entities": [],
        "update_rationale": "Query apenas de leitura."
    },
    {
        "query_name": "Q2_SimpleSearch",
        "updated_entities": [],
        "update_rationale": "Query apenas de leitura."
    },
    {
        "query_name": "Q3_AddEntitiesAndRelationships",
        "updated_entities": ["Person", "Role"],
        "update_rationale": (
            "A query insere uma nova Person e um novo Role. "
            "WatchItem é apenas referenciado para ligar a nova relação."
        )
    },
    {
        "query_name": "Q4_Recommendation",
        "updated_entities": [],
        "update_rationale": "Query apenas de leitura."
    },
    {
        "query_name": "Q5_AllPersonsOfTypeForWatchItem",
        "updated_entities": [],
        "update_rationale": "Query apenas de leitura."
    },
])

print("Entidades explicitamente modificadas por query:")
display(query_update_map_df)

Entidades explicitamente modificadas por query:


,query_name,updated_entities,update_rationale
0,Q1_Login,[],Query apenas de leitura.
1,Q2_SimpleSearch,[],Query apenas de leitura.
2,Q3_AddEntitiesAndRelationships,"[Person, Role]",A query insere uma nova Person e um novo Role....
3,Q4_Recommendation,[],Query apenas de leitura.
4,Q5_AllPersonsOfTypeForWatchItem,[],Query apenas de leitura.


In [22]:
# =========================================================
# BLOCO V15 — CALCULAR UPDATE VOLATILITY POR ENTIDADE
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "imdb_scenario",
    "query_update_map_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Definir pesos/frequências do workload
# Neste primeiro experimento, vamos assumir peso igual
# para todas as queries.
# ---------------------------------------------------------
query_frequency_df = pd.DataFrame([
    {"query_name": q["name"], "query_weight": 1.0}
    for q in imdb_scenario.workload_queries
])

# ---------------------------------------------------------
# 3) Explodir a relação query -> entidades modificadas
# ---------------------------------------------------------
all_entities = imdb_scenario.entities

expanded_rows = []

for _, row in query_update_map_df.iterrows():
    query_name = row["query_name"]
    updated_entities = row["updated_entities"]

    for entity in all_entities:
        expanded_rows.append({
            "query_name": query_name,
            "entity": entity,
            "writes_entity": 1 if entity in updated_entities else 0
        })

query_entity_write_df = pd.DataFrame(expanded_rows)

# Junta com os pesos
query_entity_write_df = query_entity_write_df.merge(
    query_frequency_df,
    on="query_name",
    how="left"
)

print("Matriz query-entidade de escrita:")
display(query_entity_write_df.head(20))

# ---------------------------------------------------------
# 4) Calcular upd(e)
# ---------------------------------------------------------
total_weight = query_frequency_df["query_weight"].sum()

entity_update_volatility_df = (
    query_entity_write_df
    .groupby("entity", as_index=False)
    .apply(
        lambda g: pd.Series({
            "update_volatility_score": (
                (g["writes_entity"] * g["query_weight"]).sum() / total_weight
            )
        })
    )
    .reset_index(drop=True)
)

entity_update_volatility_df = entity_update_volatility_df.sort_values(
    by=["update_volatility_score", "entity"],
    ascending=[False, True]
).reset_index(drop=True)

print("Update volatility por entidade:")
display(entity_update_volatility_df)

Matriz query-entidade de escrita:


,query_name,entity,writes_entity,query_weight
0,Q1_Login,Person,0,1.0
1,Q1_Login,User,0,1.0
2,Q1_Login,Genre,0,1.0
3,Q1_Login,Role,0,1.0
4,Q1_Login,Rate,0,1.0
5,Q1_Login,WatchItem,0,1.0
6,Q1_Login,Series,0,1.0
7,Q1_Login,Movie,0,1.0
8,Q1_Login,Episode,0,1.0
9,Q2_SimpleSearch,Person,0,1.0


Update volatility por entidade:


,entity,update_volatility_score
0,Person,0.2
1,Role,0.2
2,Episode,0.0
3,Genre,0.0
4,Movie,0.0
5,Rate,0.0
6,Series,0.0
7,User,0.0
8,WatchItem,0.0


In [23]:
# =========================================================
# BLOCO V16 — RESUMIR UPDATE VOLATILITY POR QUERY
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_compact_df",
    "entity_update_volatility_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Criar mapa entidade -> volatility
# ---------------------------------------------------------
entity_vol_map = dict(
    zip(
        entity_update_volatility_df["entity"],
        entity_update_volatility_df["update_volatility_score"]
    )
)

# ---------------------------------------------------------
# 3) Calcular volatilidade agregada por query
# ---------------------------------------------------------
def summarize_query_volatility(entities_touched):
    scores = [entity_vol_map.get(entity, 0.0) for entity in entities_touched]

    return pd.Series({
        "query_avg_update_volatility": float(np.mean(scores)) if scores else 0.0,
        "query_max_update_volatility": float(np.max(scores)) if scores else 0.0,
        "query_entities_with_nonzero_volatility": int(sum(1 for s in scores if s > 0))
    })

query_volatility_df = analytical_matrix_compact_df[
    ["query_name", "Qi_entities_touched"]
].copy()

volatility_summary = query_volatility_df["Qi_entities_touched"].apply(
    summarize_query_volatility
)

query_volatility_df = pd.concat([query_volatility_df, volatility_summary], axis=1)

print("Resumo de update volatility por query:")
display(query_volatility_df)

Resumo de update volatility por query:


,query_name,Qi_entities_touched,query_avg_update_volatility,query_max_update_volatility,query_entities_with_nonzero_volatility
0,Q1_Login,[User],0.000000,0.0,0.0
1,Q2_SimpleSearch,[WatchItem],0.000000,0.0,0.0
2,Q3_AddEntitiesAndRelationships,"[Person, WatchItem, Role]",0.133333,0.2,2.0
3,Q4_Recommendation,"[WatchItem, Genre, Movie, Series, Episode]",0.000000,0.0,0.0
4,Q5_AllPersonsOfTypeForWatchItem,"[WatchItem, Person, Role]",0.133333,0.2,2.0


In [24]:
# =========================================================
# BLOCO V17 — INTEGRAR UPDATE VOLATILITY À MATRIZ ANALÍTICA
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_compact_df",
    "query_volatility_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Juntar a volatilidade por query à matriz compacta
# ---------------------------------------------------------
analytical_matrix_with_volatility_df = analytical_matrix_compact_df.merge(
    query_volatility_df[
        [
            "query_name",
            "query_avg_update_volatility",
            "query_max_update_volatility",
            "query_entities_with_nonzero_volatility"
        ]
    ],
    on="query_name",
    how="left"
)

# ---------------------------------------------------------
# 3) Criar uma leitura qualitativa simples da volatilidade
# Esta classificação é apenas interpretativa, para facilitar
# a leitura da matriz. Os limiares ainda não são "regras finais".
# ---------------------------------------------------------
def classify_query_volatility(avg_vol):
    if avg_vol == 0:
        return "No volatility exposure"
    elif avg_vol <= 0.15:
        return "Low volatility exposure"
    elif avg_vol <= 0.30:
        return "Moderate volatility exposure"
    else:
        return "High volatility exposure"

analytical_matrix_with_volatility_df["query_volatility_class"] = (
    analytical_matrix_with_volatility_df["query_avg_update_volatility"]
    .apply(classify_query_volatility)
)

# ---------------------------------------------------------
# 4) Ordenar para leitura
# ---------------------------------------------------------
analytical_matrix_with_volatility_df = analytical_matrix_with_volatility_df.sort_values(
    by="query_name"
).reset_index(drop=True)

print("Matriz analítica compacta do IMDb com update volatility:")
display(analytical_matrix_with_volatility_df)

Matriz analítica compacta do IMDb com update volatility:


,query_name,query_type,Qi_entities_touched,abstract_query,Qi_n_entities_touched,Rc,Rc_selected_edges,selected_root,root_decision_rationale,D_value,...,n_containment,selected_document_depth,selected_Re,selected_DeltaR,selected_DeltaR_ratio,selected_structural_coverage,query_avg_update_volatility,query_max_update_volatility,query_entities_with_nonzero_volatility,query_volatility_class
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1,0,[],User,A query toca apenas User; User é a raiz natural.,0,...,0,0,0,0,NaN,No structural traversal,0.000000,0.0,0.0,No volatility exposure
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1,0,[],WatchItem,A query toca apenas WatchItem; WatchItem é a r...,0,...,0,0,0,0,NaN,No structural traversal,0.000000,0.0,0.0,No volatility exposure
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,...,0,2,0,2,1.0,Full absorption,0.133333,0.2,2.0,Low volatility exposure
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,4,"[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem,WatchItem é claramente a entidade central da q...,1,...,0,1,0,4,1.0,Full absorption,0.000000,0.0,0.0,No volatility exposure
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,...,0,2,0,2,1.0,Full absorption,0.133333,0.2,2.0,Low volatility exposure


In [25]:
# =========================================================
# BLOCO V18 — EXTRAIR SHAREDNESS OBSERVADA NO DATASET IMDb
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "imdb_data_bundle"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Recuperar tabelas do dataset sintético
# ---------------------------------------------------------
users_df = imdb_data_bundle.tables["users"]
persons_df = imdb_data_bundle.tables["persons"]
genres_df = imdb_data_bundle.tables["genres"]
watchitems_df = imdb_data_bundle.tables["watchitems"]
movies_df = imdb_data_bundle.tables["movies"]
series_df = imdb_data_bundle.tables["series"]
episodes_df = imdb_data_bundle.tables["episodes"]
roles_df = imdb_data_bundle.tables["roles"]
rates_df = imdb_data_bundle.tables["rates"]

# ---------------------------------------------------------
# 3) Função auxiliar para calcular sharedness observada
# root_col   = coluna que representa a instância da raiz
# target_col = coluna que representa a instância da entidade candidata
# ---------------------------------------------------------
def summarize_observed_sharedness(
    df: pd.DataFrame,
    root_col: str,
    target_col: str,
    root_entity: str,
    target_entity: str,
    relationship_context: str
) -> dict:
    """
    Calcula a sharedness observada:
    - média de raízes distintas por instância target
    - máximo de raízes distintas por instância target

    Interpretação:
    - se avg_roots_per_target = 1, a entidade target é local a uma raiz
    - se > 1, a entidade target é compartilhada entre múltiplas raízes
    """
    # Agrupa por instância target e conta em quantas raízes ela aparece
    target_to_roots = df.groupby(target_col)[root_col].nunique()

    return {
        "root_entity": root_entity,
        "target_entity": target_entity,
        "relationship_context": relationship_context,
        "n_target_instances_observed": int(target_to_roots.shape[0]),
        "avg_roots_per_target": round(float(target_to_roots.mean()), 2),
        "max_roots_per_target": int(target_to_roots.max())
    }

# ---------------------------------------------------------
# 4) Extrair sharedness para pares relevantes do IMDb
# ---------------------------------------------------------
sharedness_rows = []

# Caso trivial: User dentro de User
sharedness_rows.append(
    summarize_observed_sharedness(
        df=users_df.assign(root_id=users_df["user_id"], target_id=users_df["user_id"]),
        root_col="root_id",
        target_col="target_id",
        root_entity="User",
        target_entity="User",
        relationship_context="Identity / local entity"
    )
)

# Caso trivial: WatchItem dentro de WatchItem
sharedness_rows.append(
    summarize_observed_sharedness(
        df=watchitems_df.assign(
            root_id=watchitems_df["watchitem_id"],
            target_id=watchitems_df["watchitem_id"]
        ),
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="WatchItem",
        relationship_context="Identity / local entity"
    )
)

# WatchItem -> Role
sharedness_rows.append(
    summarize_observed_sharedness(
        df=roles_df,
        root_col="watchitem_id",
        target_col="role_id",
        root_entity="WatchItem",
        target_entity="Role",
        relationship_context="WatchItem to Role"
    )
)

# WatchItem -> Person (via Role)
sharedness_rows.append(
    summarize_observed_sharedness(
        df=roles_df,
        root_col="watchitem_id",
        target_col="person_id",
        root_entity="WatchItem",
        target_entity="Person",
        relationship_context="WatchItem to Person via Role"
    )
)

# WatchItem -> Genre
sharedness_rows.append(
    summarize_observed_sharedness(
        df=watchitems_df,
        root_col="watchitem_id",
        target_col="genre_id",
        root_entity="WatchItem",
        target_entity="Genre",
        relationship_context="WatchItem to Genre"
    )
)

# WatchItem -> Movie
sharedness_rows.append(
    summarize_observed_sharedness(
        df=movies_df.assign(root_id=movies_df["watchitem_id"], target_id=movies_df["watchitem_id"]),
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Movie",
        relationship_context="Subtype identity: Movie is WatchItem"
    )
)

# WatchItem -> Series
sharedness_rows.append(
    summarize_observed_sharedness(
        df=series_df.assign(root_id=series_df["watchitem_id"], target_id=series_df["watchitem_id"]),
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Series",
        relationship_context="Subtype identity: Series is WatchItem"
    )
)

# WatchItem -> Episode
sharedness_rows.append(
    summarize_observed_sharedness(
        df=episodes_df.assign(root_id=episodes_df["watchitem_id"], target_id=episodes_df["watchitem_id"]),
        root_col="root_id",
        target_col="target_id",
        root_entity="WatchItem",
        target_entity="Episode",
        relationship_context="Subtype identity: Episode is WatchItem"
    )
)

# Série -> Episode (referência importante para containment)
episodes_with_series = episodes_df.dropna(subset=["series_watchitem_id"]).copy()

if not episodes_with_series.empty:
    sharedness_rows.append(
        summarize_observed_sharedness(
            df=episodes_with_series,
            root_col="series_watchitem_id",
            target_col="watchitem_id",
            root_entity="Series",
            target_entity="Episode",
            relationship_context="Series to Episode (containment-like)"
        )
    )

sharedness_df = pd.DataFrame(sharedness_rows)

print("Sharedness observada no dataset IMDb sintético:")
display(sharedness_df)

Sharedness observada no dataset IMDb sintético:


,root_entity,target_entity,relationship_context,n_target_instances_observed,avg_roots_per_target,max_roots_per_target
0,User,User,Identity / local entity,100,1.00,1
1,WatchItem,WatchItem,Identity / local entity,80,1.00,1
2,WatchItem,Role,WatchItem to Role,290,1.00,1
3,WatchItem,Person,WatchItem to Person via Role,114,2.54,9
4,WatchItem,Genre,WatchItem to Genre,10,8.00,12
5,WatchItem,Movie,Subtype identity: Movie is WatchItem,34,1.00,1
6,WatchItem,Series,Subtype identity: Series is WatchItem,15,1.00,1
7,WatchItem,Episode,Subtype identity: Episode is WatchItem,31,1.00,1
8,Series,Episode,Series to Episode (containment-like),31,1.00,1


In [26]:
# =========================================================
# BLOCO V19 — RESUMIR SHAREDNESS POR QUERY
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_with_volatility_df",
    "sharedness_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Criar um mapa (root_entity, target_entity) -> sharedness média/máxima
# ---------------------------------------------------------
sharedness_map = {}

for _, row in sharedness_df.iterrows():
    key = (row["root_entity"], row["target_entity"])
    sharedness_map[key] = {
        "avg_roots_per_target": row["avg_roots_per_target"],
        "max_roots_per_target": row["max_roots_per_target"],
        "relationship_context": row["relationship_context"]
    }

# ---------------------------------------------------------
# 3) Função auxiliar para resumir sharedness de uma query
# ---------------------------------------------------------
def summarize_query_sharedness(selected_root, entities_touched):
    """
    Resume a sharedness dos alvos tocados pela query em relação à raiz escolhida.

    Observação:
    - por padrão, excluímos a própria raiz do cálculo agregado
      para focar nas entidades candidatas a embutimento.
    """
    candidate_entities = [e for e in entities_touched if e != selected_root]

    avg_scores = []
    max_scores = []
    details = []

    for entity in candidate_entities:
        key = (selected_root, entity)
        info = sharedness_map.get(key, None)

        if info is not None:
            avg_scores.append(info["avg_roots_per_target"])
            max_scores.append(info["max_roots_per_target"])
            details.append({
                "target_entity": entity,
                "avg_roots_per_target": info["avg_roots_per_target"],
                "max_roots_per_target": info["max_roots_per_target"],
                "relationship_context": info["relationship_context"]
            })
        else:
            details.append({
                "target_entity": entity,
                "avg_roots_per_target": None,
                "max_roots_per_target": None,
                "relationship_context": "No observed mapping available"
            })

    return pd.Series({
        "query_avg_sharedness": float(np.mean(avg_scores)) if avg_scores else np.nan,
        "query_max_sharedness": float(np.max(max_scores)) if max_scores else np.nan,
        "query_entities_with_sharedness_gt_1": int(sum(1 for x in avg_scores if x > 1)),
        "query_sharedness_details": details
    })

# ---------------------------------------------------------
# 4) Aplicar para cada query da matriz compacta
# ---------------------------------------------------------
query_sharedness_df = analytical_matrix_with_volatility_df[
    ["query_name", "selected_root", "Qi_entities_touched"]
].copy()

sharedness_summary = query_sharedness_df.apply(
    lambda row: summarize_query_sharedness(
        selected_root=row["selected_root"],
        entities_touched=row["Qi_entities_touched"]
    ),
    axis=1
)

query_sharedness_df = pd.concat([query_sharedness_df, sharedness_summary], axis=1)

# ---------------------------------------------------------
# 5) Classificação qualitativa simples
# Observação:
# limiares apenas interpretativos por enquanto
# ---------------------------------------------------------
def classify_sharedness(avg_sharedness):
    if pd.isna(avg_sharedness):
        return "Not applicable"
    elif avg_sharedness <= 1.0:
        return "Low sharedness"
    elif avg_sharedness <= 2.0:
        return "Moderate sharedness"
    else:
        return "High sharedness"

query_sharedness_df["query_sharedness_class"] = query_sharedness_df[
    "query_avg_sharedness"
].apply(classify_sharedness)

print("Resumo de sharedness por query:")
display(query_sharedness_df)

Resumo de sharedness por query:


,query_name,selected_root,Qi_entities_touched,query_avg_sharedness,query_max_sharedness,query_entities_with_sharedness_gt_1,query_sharedness_details,query_sharedness_class
0,Q1_Login,User,[User],NaN,NaN,0,[],Not applicable
1,Q2_SimpleSearch,WatchItem,[WatchItem],NaN,NaN,0,[],Not applicable
2,Q3_AddEntitiesAndRelationships,WatchItem,"[Person, WatchItem, Role]",1.77,9.0,1,"[{'target_entity': 'Person', 'avg_roots_per_ta...",Moderate sharedness
3,Q4_Recommendation,WatchItem,"[WatchItem, Genre, Movie, Series, Episode]",2.75,12.0,1,"[{'target_entity': 'Genre', 'avg_roots_per_tar...",High sharedness
4,Q5_AllPersonsOfTypeForWatchItem,WatchItem,"[WatchItem, Person, Role]",1.77,9.0,1,"[{'target_entity': 'Person', 'avg_roots_per_ta...",Moderate sharedness


In [27]:
# =========================================================
# BLOCO V20 — INTEGRAR SHAREDNESS À MATRIZ ANALÍTICA FINAL
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_with_volatility_df",
    "query_sharedness_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Juntar sharedness à matriz
# ---------------------------------------------------------
analytical_matrix_final_df = analytical_matrix_with_volatility_df.merge(
    query_sharedness_df[
        [
            "query_name",
            "query_avg_sharedness",
            "query_max_sharedness",
            "query_entities_with_sharedness_gt_1",
            "query_sharedness_class",
            "query_sharedness_details"
        ]
    ],
    on="query_name",
    how="left"
)

# ---------------------------------------------------------
# 3) Criar uma leitura qualitativa combinada
# Esta coluna é apenas interpretativa por enquanto.
# ---------------------------------------------------------
def combined_document_readiness(row):
    coverage = row["selected_structural_coverage"]
    volatility = row["query_volatility_class"]
    sharedness = row["query_sharedness_class"]

    if coverage == "No structural traversal":
        return "Structurally neutral for document"

    if coverage == "Full absorption":
        if volatility == "No volatility exposure" and sharedness in ["Low sharedness", "Not applicable"]:
            return "Strong document candidate"
        elif volatility == "No volatility exposure" and sharedness in ["Moderate sharedness", "High sharedness"]:
            return "Promising document candidate with duplication trade-off"
        elif volatility in ["Low volatility exposure", "Moderate volatility exposure"] and sharedness in ["Low sharedness", "Moderate sharedness"]:
            return "Document candidate with maintenance trade-off"
        else:
            return "Document candidate with stronger trade-offs"

    if coverage == "Partial absorption":
        return "Partial document candidate"

    return "Weak document candidate"

analytical_matrix_final_df["document_candidate_assessment"] = analytical_matrix_final_df.apply(
    combined_document_readiness,
    axis=1
)

# ---------------------------------------------------------
# 4) Ordenar e exibir
# ---------------------------------------------------------
analytical_matrix_final_df = analytical_matrix_final_df.sort_values(
    by="query_name"
).reset_index(drop=True)

print("Matriz analítica final do IMDb para o modelo documento:")
display(analytical_matrix_final_df)

Matriz analítica final do IMDb para o modelo documento:


,query_name,query_type,Qi_entities_touched,abstract_query,Qi_n_entities_touched,Rc,Rc_selected_edges,selected_root,root_decision_rationale,D_value,...,query_avg_update_volatility,query_max_update_volatility,query_entities_with_nonzero_volatility,query_volatility_class,query_avg_sharedness,query_max_sharedness,query_entities_with_sharedness_gt_1,query_sharedness_class,query_sharedness_details,document_candidate_assessment
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1,0,[],User,A query toca apenas User; User é a raiz natural.,0,...,0.000000,0.0,0.0,No volatility exposure,NaN,NaN,0,Not applicable,[],Structurally neutral for document
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1,0,[],WatchItem,A query toca apenas WatchItem; WatchItem é a r...,0,...,0.000000,0.0,0.0,No volatility exposure,NaN,NaN,0,Not applicable,[],Structurally neutral for document
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,...,0.133333,0.2,2.0,Low volatility exposure,1.77,9.0,1,Moderate sharedness,"[{'target_entity': 'Person', 'avg_roots_per_ta...",Document candidate with maintenance trade-off
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,4,"[Genre -- WatchItem [has_genre], Series -- Wat...",WatchItem,WatchItem é claramente a entidade central da q...,1,...,0.000000,0.0,0.0,No volatility exposure,2.75,12.0,1,High sharedness,"[{'target_entity': 'Genre', 'avg_roots_per_tar...",Promising document candidate with duplication ...
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,2,"[Person -- Role [acts_in_via_role], Role -- Wa...",WatchItem,"Embora haja empate estrutural entre Person, Ro...",2,...,0.133333,0.2,2.0,Low volatility exposure,1.77,9.0,1,Moderate sharedness,"[{'target_entity': 'Person', 'avg_roots_per_ta...",Document candidate with maintenance trade-off


In [28]:
# =========================================================
# BLOCO V21A — MATRIZ FINAL DE VARIÁVEIS CALCULADAS
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "analytical_matrix_final_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Selecionar as colunas principais da análise documental
# Esta será a matriz-base para justificar as configurações
# MongoDB que vamos construir.
# ---------------------------------------------------------
document_variable_matrix_df = analytical_matrix_final_df[
    [
        "query_name",
        "query_type",
        "Qi_entities_touched",
        "abstract_query",
        "Qi_n_entities_touched",
        "selected_root",
        "Rc",
        "D_value",
        "selected_document_depth",
        "selected_Re",
        "selected_DeltaR",
        "selected_DeltaR_ratio",
        "selected_structural_coverage",
        "n_association",
        "n_associative",
        "n_containment",
        "query_avg_update_volatility",
        "query_max_update_volatility",
        "query_entities_with_nonzero_volatility",
        "query_volatility_class",
        "query_avg_sharedness",
        "query_max_sharedness",
        "query_entities_with_sharedness_gt_1",
        "query_sharedness_class",
        "document_candidate_assessment"
    ]
].copy()

# ---------------------------------------------------------
# 3) Criar colunas auxiliares para leitura
# ---------------------------------------------------------
document_variable_matrix_df["has_full_absorption"] = (
    document_variable_matrix_df["selected_structural_coverage"] == "Full absorption"
)

document_variable_matrix_df["is_structurally_neutral"] = (
    document_variable_matrix_df["selected_structural_coverage"] == "No structural traversal"
)

# Ordena por nome da query
document_variable_matrix_df = document_variable_matrix_df.sort_values(
    by="query_name"
).reset_index(drop=True)

print("Matriz final de variáveis calculadas do IMDb:")
display(document_variable_matrix_df)

Matriz final de variáveis calculadas do IMDb:


,query_name,query_type,Qi_entities_touched,abstract_query,Qi_n_entities_touched,selected_root,Rc,D_value,selected_document_depth,selected_Re,...,query_max_update_volatility,query_entities_with_nonzero_volatility,query_volatility_class,query_avg_sharedness,query_max_sharedness,query_entities_with_sharedness_gt_1,query_sharedness_class,document_candidate_assessment,has_full_absorption,is_structurally_neutral
0,Q1_Login,select,[User],RETURN password FROM User WHERE username = ?,1,User,0,0,0,0,...,0.0,0.0,No volatility exposure,NaN,NaN,0,Not applicable,Structurally neutral for document,False,True
1,Q2_SimpleSearch,select,[WatchItem],RETURN ALL FROM WatchItem WHERE title = ?,1,WatchItem,0,0,0,0,...,0.0,0.0,No volatility exposure,NaN,NaN,0,Not applicable,Structurally neutral for document,False,True
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",INSERT Person and connect Person with WatchItem,3,WatchItem,2,2,2,0,...,0.2,2.0,Low volatility exposure,1.77,9.0,1,Moderate sharedness,Document candidate with maintenance trade-off,True,False
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",Recommendation query by genre and type,5,WatchItem,4,1,1,0,...,0.0,0.0,No volatility exposure,2.75,12.0,1,High sharedness,Promising document candidate with duplication ...,True,False
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]","RETURN Person.ALL FROM WatchItem, Person WHERE...",3,WatchItem,2,2,2,0,...,0.2,2.0,Low volatility exposure,1.77,9.0,1,Moderate sharedness,Document candidate with maintenance trade-off,True,False


In [29]:
# =========================================================
# BLOCO V21B — MATRIZ DE DERIVAÇÃO DAS CONFIGURAÇÕES M1–M4
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "document_variable_matrix_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro o bloco V21A."
        )

# ---------------------------------------------------------
# 2) Selecionar queries que motivam cada configuração
# Observação:
# aqui ainda estamos documentando a origem metodológica
# das hipóteses M1–M4.
# ---------------------------------------------------------

# Q4 -> hipótese de WatchItem raso
q4_row = document_variable_matrix_df[
    document_variable_matrix_df["query_name"] == "Q4_Recommendation"
].iloc[0]

# Q3 -> hipótese com Role
q3_row = document_variable_matrix_df[
    document_variable_matrix_df["query_name"] == "Q3_AddEntitiesAndRelationships"
].iloc[0]

# Q5 -> hipótese com Role + Person
q5_row = document_variable_matrix_df[
    document_variable_matrix_df["query_name"] == "Q5_AllPersonsOfTypeForWatchItem"
].iloc[0]

# ---------------------------------------------------------
# 3) Criar a matriz de derivação das configurações
# ---------------------------------------------------------
mongo_design_derivation_df = pd.DataFrame([
    {
        "design_id": "M1",
        "design_name": "watchitem_shallow",
        "derived_from_queries": ["Q4_Recommendation"],
        "selected_root": "WatchItem",
        "proposed_embedding_depth": 1,
        "proposed_embedded_entities": ["Genre", "Movie", "Series", "Episode"],
        "proposed_referenced_entities": ["Role", "Person"],
        "source_Rc": int(q4_row["Rc"]),
        "source_D": int(q4_row["D_value"]),
        "source_DeltaR_ratio": float(q4_row["selected_DeltaR_ratio"]),
        "source_relationship_profile": {
            "association": int(q4_row["n_association"]),
            "associative": int(q4_row["n_associative"]),
            "containment": int(q4_row["n_containment"])
        },
        "source_volatility_class": q4_row["query_volatility_class"],
        "source_sharedness_class": q4_row["query_sharedness_class"],
        "derivation_rationale": (
            "Derived from Q4, which presents the clearest document-oriented profile: "
            "high Rc, shallow D, full structural absorption at depth 1, and no volatility exposure."
        )
    },
    {
        "design_id": "M2",
        "design_name": "watchitem_with_role",
        "derived_from_queries": ["Q3_AddEntitiesAndRelationships", "Q5_AllPersonsOfTypeForWatchItem"],
        "selected_root": "WatchItem",
        "proposed_embedding_depth": 1,
        "proposed_embedded_entities": ["Role"],
        "proposed_referenced_entities": ["Person", "Genre", "Movie", "Series", "Episode"],
        "source_Rc": int(q3_row["Rc"]),
        "source_D": 1,
        "source_DeltaR_ratio": 0.5,
        "source_relationship_profile": {
            "association": int(q3_row["n_association"]),
            "associative": int(q3_row["n_associative"]),
            "containment": int(q3_row["n_containment"])
        },
        "source_volatility_class": q3_row["query_volatility_class"],
        "source_sharedness_class": q3_row["query_sharedness_class"],
        "derivation_rationale": (
            "Derived from Q3/Q5 as a conservative document hypothesis: Role is embedded because it has "
            "low sharedness relative to WatchItem, while Person remains referenced."
        )
    },
    {
        "design_id": "M3",
        "design_name": "watchitem_with_role_person",
        "derived_from_queries": ["Q3_AddEntitiesAndRelationships", "Q5_AllPersonsOfTypeForWatchItem"],
        "selected_root": "WatchItem",
        "proposed_embedding_depth": 2,
        "proposed_embedded_entities": ["Role", "Person"],
        "proposed_referenced_entities": ["Genre", "Movie", "Series", "Episode"],
        "source_Rc": int(q5_row["Rc"]),
        "source_D": int(q5_row["D_value"]),
        "source_DeltaR_ratio": float(q5_row["selected_DeltaR_ratio"]),
        "source_relationship_profile": {
            "association": int(q5_row["n_association"]),
            "associative": int(q5_row["n_associative"]),
            "containment": int(q5_row["n_containment"])
        },
        "source_volatility_class": q5_row["query_volatility_class"],
        "source_sharedness_class": q5_row["query_sharedness_class"],
        "derivation_rationale": (
            "Derived from Q3/Q5 as the strongest document hypothesis for full absorption: "
            "Role and Person are both embedded under WatchItem."
        )
    },
    {
        "design_id": "M4",
        "design_name": "series_with_episode",
        "derived_from_queries": ["ContainmentExploration"],
        "selected_root": "Series",
        "proposed_embedding_depth": 1,
        "proposed_embedded_entities": ["Episode"],
        "proposed_referenced_entities": ["WatchItem", "Genre", "Role", "Person"],
        "source_Rc": None,
        "source_D": 1,
        "source_DeltaR_ratio": None,
        "source_relationship_profile": {
            "association": 0,
            "associative": 0,
            "containment": 1
        },
        "source_volatility_class": "No volatility exposure",
        "source_sharedness_class": "Low sharedness",
        "derivation_rationale": (
            "Derived from the strongest containment-like structure in the schema: Series contains Episode, "
            "with sharedness equal to 1.0 and strong aggregate cohesion."
        )
    }
])

# ---------------------------------------------------------
# 4) Criar versões amigáveis para visualização
# ---------------------------------------------------------
def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

mongo_design_derivation_view_df = mongo_design_derivation_df.copy()

for col in ["derived_from_queries", "proposed_embedded_entities", "proposed_referenced_entities"]:
    mongo_design_derivation_view_df[col] = mongo_design_derivation_view_df[col].apply(list_to_str)

print("Matriz de derivação das configurações documentais M1–M4:")
display(mongo_design_derivation_view_df)

Matriz de derivação das configurações documentais M1–M4:


,design_id,design_name,derived_from_queries,selected_root,proposed_embedding_depth,proposed_embedded_entities,proposed_referenced_entities,source_Rc,source_D,source_DeltaR_ratio,source_relationship_profile,source_volatility_class,source_sharedness_class,derivation_rationale
0,M1,watchitem_shallow,Q4_Recommendation,WatchItem,1,"Genre, Movie, Series, Episode","Role, Person",4.0,1,1.0,"{'association': 4, 'associative': 0, 'containm...",No volatility exposure,High sharedness,"Derived from Q4, which presents the clearest d..."
1,M2,watchitem_with_role,"Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",WatchItem,1,Role,"Person, Genre, Movie, Series, Episode",2.0,1,0.5,"{'association': 0, 'associative': 2, 'containm...",Low volatility exposure,Moderate sharedness,Derived from Q3/Q5 as a conservative document ...
2,M3,watchitem_with_role_person,"Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",WatchItem,2,"Role, Person","Genre, Movie, Series, Episode",2.0,2,1.0,"{'association': 0, 'associative': 2, 'containm...",Low volatility exposure,Moderate sharedness,Derived from Q3/Q5 as the strongest document h...
3,M4,series_with_episode,ContainmentExploration,Series,1,Episode,"WatchItem, Genre, Role, Person",NaN,1,NaN,"{'association': 0, 'associative': 0, 'containm...",No volatility exposure,Low sharedness,Derived from the strongest containment-like st...


In [30]:
# =========================================================
# BLOCO V22 — CATÁLOGO EXPANDIDO DE EXPERIMENTOS MONGODB
# =========================================================
#COM TODOS OS EXPERIMENTOS QUE SERÃO REALIZADOS. POR SF. E CADA CONFIGURAÇÃO DIFERENTE PARA O MONGODB.



import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "mongo_design_derivation_df"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro o bloco que cria "
            "a matriz de derivação das configurações M1–M4."
        )

# ---------------------------------------------------------
# 2) Função auxiliar para deixar listas em string
# ---------------------------------------------------------
def list_to_str(values):
    if isinstance(values, list):
        return ", ".join(str(v) for v in values)
    return str(values)

# ---------------------------------------------------------
# 3) Definir os scale factors planejados
# Observação:
# - SF1 será a fase inicial do experimento
# - SF10, SF30 e SF100 ficam preparados para a continuidade
# ---------------------------------------------------------
mongo_scale_factors_df = pd.DataFrame([
    {
        "scale_factor": 1,
        "scale_label": "sf1",
        "execution_phase": "current"
    },
    {
        "scale_factor": 10,
        "scale_label": "sf10",
        "execution_phase": "planned"
    },
    {
        "scale_factor": 30,
        "scale_label": "sf30",
        "execution_phase": "planned"
    },
    {
        "scale_factor": 100,
        "scale_label": "sf100",
        "execution_phase": "planned"
    }
])

print("Scale factors planejados:")
display(mongo_scale_factors_df)

# ---------------------------------------------------------
# 4) Expandir cada design documental para cada scale factor
# Regra:
# - uma linha por (design_id, scale_factor)
# - cada linha representa um banco MongoDB separado
# ---------------------------------------------------------
mongo_experiment_rows = []

for _, design_row in mongo_design_derivation_df.iterrows():
    for _, sf_row in mongo_scale_factors_df.iterrows():
        design_id = design_row["design_id"]
        design_name = design_row["design_name"]
        scale_factor = sf_row["scale_factor"]
        scale_label = sf_row["scale_label"]
        execution_phase = sf_row["execution_phase"]

        # Nome do banco MongoDB
        mongo_db_name = f"imdb_mongo_{scale_label}_{design_id.lower()}_{design_name}"

        mongo_experiment_rows.append({
            "experiment_id": f"{design_id}_{scale_label.upper()}",
            "design_id": design_id,
            "design_name": design_name,
            "scale_factor": scale_factor,
            "scale_label": scale_label,
            "execution_phase": execution_phase,
            "mongo_db_name": mongo_db_name,

            # Estrutura documental
            "selected_root": design_row["selected_root"],
            "embedding_depth": design_row["proposed_embedding_depth"],
            "embedded_entities": design_row["proposed_embedded_entities"],
            "referenced_entities": design_row["proposed_referenced_entities"],

            # Motivação analítica
            "derived_from_queries": design_row["derived_from_queries"],
            "source_Rc": design_row["source_Rc"],
            "source_D": design_row["source_D"],
            "source_DeltaR_ratio": design_row["source_DeltaR_ratio"],
            "source_volatility_class": design_row["source_volatility_class"],
            "source_sharedness_class": design_row["source_sharedness_class"],

            # Descrição do experimento
            "primary_collection": (
                "watchitems" if design_row["selected_root"] == "WatchItem"
                else "series" if design_row["selected_root"] == "Series"
                else design_row["selected_root"].lower()
            ),
            "experiment_goal": design_row["derivation_rationale"]
        })

mongo_experiment_catalog_df = pd.DataFrame(mongo_experiment_rows)

# ---------------------------------------------------------
# 5) Criar versão amigável para visualização no Jupyter
# ---------------------------------------------------------
mongo_experiment_catalog_view_df = mongo_experiment_catalog_df.copy()

for col in ["embedded_entities", "referenced_entities", "derived_from_queries"]:
    mongo_experiment_catalog_view_df[col] = mongo_experiment_catalog_view_df[col].apply(list_to_str)

# ---------------------------------------------------------
# 6) Exibir o catálogo
# ---------------------------------------------------------
print("Catálogo expandido de experimentos MongoDB:")
display(mongo_experiment_catalog_view_df)

Scale factors planejados:


,scale_factor,scale_label,execution_phase
0,1,sf1,current
1,10,sf10,planned
2,30,sf30,planned
3,100,sf100,planned


Catálogo expandido de experimentos MongoDB:


,experiment_id,design_id,design_name,scale_factor,scale_label,execution_phase,mongo_db_name,selected_root,embedding_depth,embedded_entities,referenced_entities,derived_from_queries,source_Rc,source_D,source_DeltaR_ratio,source_volatility_class,source_sharedness_class,primary_collection,experiment_goal
0,M1_SF1,M1,watchitem_shallow,1,sf1,current,imdb_mongo_sf1_m1_watchitem_shallow,WatchItem,1,"Genre, Movie, Series, Episode","Role, Person",Q4_Recommendation,4.0,1,1.0,No volatility exposure,High sharedness,watchitems,"Derived from Q4, which presents the clearest d..."
1,M1_SF10,M1,watchitem_shallow,10,sf10,planned,imdb_mongo_sf10_m1_watchitem_shallow,WatchItem,1,"Genre, Movie, Series, Episode","Role, Person",Q4_Recommendation,4.0,1,1.0,No volatility exposure,High sharedness,watchitems,"Derived from Q4, which presents the clearest d..."
2,M1_SF30,M1,watchitem_shallow,30,sf30,planned,imdb_mongo_sf30_m1_watchitem_shallow,WatchItem,1,"Genre, Movie, Series, Episode","Role, Person",Q4_Recommendation,4.0,1,1.0,No volatility exposure,High sharedness,watchitems,"Derived from Q4, which presents the clearest d..."
3,M1_SF100,M1,watchitem_shallow,100,sf100,planned,imdb_mongo_sf100_m1_watchitem_shallow,WatchItem,1,"Genre, Movie, Series, Episode","Role, Person",Q4_Recommendation,4.0,1,1.0,No volatility exposure,High sharedness,watchitems,"Derived from Q4, which presents the clearest d..."
4,M2_SF1,M2,watchitem_with_role,1,sf1,current,imdb_mongo_sf1_m2_watchitem_with_role,WatchItem,1,Role,"Person, Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,1,0.5,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as a conservative document ...
5,M2_SF10,M2,watchitem_with_role,10,sf10,planned,imdb_mongo_sf10_m2_watchitem_with_role,WatchItem,1,Role,"Person, Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,1,0.5,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as a conservative document ...
6,M2_SF30,M2,watchitem_with_role,30,sf30,planned,imdb_mongo_sf30_m2_watchitem_with_role,WatchItem,1,Role,"Person, Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,1,0.5,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as a conservative document ...
7,M2_SF100,M2,watchitem_with_role,100,sf100,planned,imdb_mongo_sf100_m2_watchitem_with_role,WatchItem,1,Role,"Person, Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,1,0.5,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as a conservative document ...
8,M3_SF1,M3,watchitem_with_role_person,1,sf1,current,imdb_mongo_sf1_m3_watchitem_with_role_person,WatchItem,2,"Role, Person","Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,2,1.0,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as the strongest document h...
9,M3_SF10,M3,watchitem_with_role_person,10,sf10,planned,imdb_mongo_sf10_m3_watchitem_with_role_person,WatchItem,2,"Role, Person","Genre, Movie, Series, Episode","Q3_AddEntitiesAndRelationships, Q5_AllPersonsO...",2.0,2,1.0,Low volatility exposure,Moderate sharedness,watchitems,Derived from Q3/Q5 as the strongest document h...


In [31]:
# =========================================================
# BLOCO V23 — SELECIONAR O EXPERIMENTO MONGODB ATIVO
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "mongo_experiment_catalog_df" not in globals():
    raise NameError(
        "O DataFrame 'mongo_experiment_catalog_df' não está definido. "
        "Rode primeiro o bloco do catálogo expandido."
    )

# ---------------------------------------------------------
# 2) Definir o experimento ativo
# Sugestão inicial:
# - começar com M1_SF1
# Depois você pode trocar facilmente para:
# - M2_SF1
# - M3_SF1
# - M4_SF1
# ---------------------------------------------------------
ACTIVE_EXPERIMENT_ID = "M1_SF1"

# ---------------------------------------------------------
# 3) Selecionar a linha correspondente
# ---------------------------------------------------------
active_experiment_df = mongo_experiment_catalog_df[
    mongo_experiment_catalog_df["experiment_id"] == ACTIVE_EXPERIMENT_ID
].copy()

if active_experiment_df.empty:
    raise ValueError(
        f"Experimento '{ACTIVE_EXPERIMENT_ID}' não encontrado no catálogo."
    )

if len(active_experiment_df) != 1:
    raise ValueError(
        f"Experimento '{ACTIVE_EXPERIMENT_ID}' retornou mais de uma linha. "
        "O catálogo deveria ter IDs únicos."
    )

active_experiment_row = active_experiment_df.iloc[0].to_dict()

# ---------------------------------------------------------
# 4) Exibir a especificação ativa
# ---------------------------------------------------------
print("Experimento MongoDB ativo:")
for key, value in active_experiment_row.items():
    print(f"{key}: {value}")

Experimento MongoDB ativo:
experiment_id: M1_SF1
design_id: M1
design_name: watchitem_shallow
scale_factor: 1
scale_label: sf1
execution_phase: current
mongo_db_name: imdb_mongo_sf1_m1_watchitem_shallow
selected_root: WatchItem
embedding_depth: 1
embedded_entities: ['Genre', 'Movie', 'Series', 'Episode']
referenced_entities: ['Role', 'Person']
derived_from_queries: ['Q4_Recommendation']
source_Rc: 4.0
source_D: 1
source_DeltaR_ratio: 1.0
source_volatility_class: No volatility exposure
source_sharedness_class: High sharedness
primary_collection: watchitems
experiment_goal: Derived from Q4, which presents the clearest document-oriented profile: high Rc, shallow D, full structural absorption at depth 1, and no volatility exposure.


In [32]:
# =========================================================
# BLOCO V24 — ESTRUTURA DO PAYLOAD MONGODB
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List, Any
import pandas as pd

@dataclass
class MongoExperimentPayload:
    experiment_id: str
    mongo_db_name: str
    selected_root: str
    primary_collection: str
    collections: Dict[str, List[Dict[str, Any]]] = field(default_factory=dict)

    def collection_names(self) -> List[str]:
        return list(self.collections.keys())

    def summary(self) -> pd.DataFrame:
        rows = []

        for collection_name, docs in self.collections.items():
            sample_keys = sorted(list(docs[0].keys())) if len(docs) > 0 else []

            rows.append({
                "collection_name": collection_name,
                "n_documents": len(docs),
                "sample_top_level_keys": sample_keys
            })

        return pd.DataFrame(rows).sort_values("collection_name").reset_index(drop=True)

In [51]:
# =========================================================
# BLOCO V25 — GERAR PAYLOAD MONGODB FULL PARA M1_SF1
# =========================================================

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "imdb_data_bundle",
    "active_experiment_row",
    "MongoExperimentPayload"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Funções auxiliares para converter tipos pandas/numpy
# em tipos nativos do Python
# ---------------------------------------------------------
def to_python_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value

def row_to_python_dict(row: pd.Series) -> dict:
    return {
        key: to_python_value(value)
        for key, value in row.to_dict().items()
    }

# ---------------------------------------------------------
# 3) Função específica para construir o payload FULL do M1
# M1 = watchitem_shallow
#
# Diferença em relação à versão anterior:
# - a coleção principal continua otimizada
# - mas agora o banco contém TODO o domínio
# ---------------------------------------------------------
def build_m1_watchitem_shallow_full_payload(
    bundle,
    experiment_row: dict
) -> MongoExperimentPayload:
    """
    Constrói o payload FULL da configuração M1:
    - coleção principal otimizada: watchitems
      com Genre + subtipo embutidos
    - restante do domínio também permanece no banco
      em coleções próprias, para manter a comparação justa
    """

    if experiment_row["design_id"] != "M1":
        raise ValueError(
            "Esta função foi feita especificamente para o design M1."
        )

    # -----------------------------------------------------
    # Recuperar tabelas do dataset sintético
    # -----------------------------------------------------
    users_df = bundle.tables["users"].copy()
    persons_df = bundle.tables["persons"].copy()
    genres_df = bundle.tables["genres"].copy()
    watchitems_df = bundle.tables["watchitems"].copy()
    movies_df = bundle.tables["movies"].copy()
    series_df = bundle.tables["series"].copy()
    episodes_df = bundle.tables["episodes"].copy()
    roles_df = bundle.tables["roles"].copy()
    rates_df = bundle.tables["rates"].copy()

    # -----------------------------------------------------
    # Criar mapas auxiliares para o embutimento na coleção
    # principal watchitems
    # -----------------------------------------------------
    genres_map = {
        int(row["genre_id"]): row_to_python_dict(row)
        for _, row in genres_df.iterrows()
    }

    movies_map = {
        int(row["watchitem_id"]): row_to_python_dict(row)
        for _, row in movies_df.iterrows()
    }

    series_map = {
        int(row["watchitem_id"]): row_to_python_dict(row)
        for _, row in series_df.iterrows()
    }

    episodes_map = {
        int(row["watchitem_id"]): row_to_python_dict(row)
        for _, row in episodes_df.iterrows()
    }

    # -----------------------------------------------------
    # Construir a coleção principal: watchitems
    # M1 continua sendo um design "shallow":
    # - embute Genre
    # - embute subtipo correspondente
    # - NÃO embute Role nem Person
    # -----------------------------------------------------
    watchitem_documents = []

    for _, row in watchitems_df.iterrows():
        doc = row_to_python_dict(row)

        watchitem_id = int(doc["watchitem_id"])
        genre_id = doc.get("genre_id", None)

        # Embute gênero
        doc["genre"] = (
            genres_map.get(int(genre_id), None)
            if genre_id is not None else None
        )

        # Embute subtipo correspondente
        doc["movie_data"] = movies_map.get(watchitem_id, None)
        doc["series_data"] = series_map.get(watchitem_id, None)
        doc["episode_data"] = episodes_map.get(watchitem_id, None)

        # Campo auxiliar para indicar o subtipo documental observado
        if doc["movie_data"] is not None:
            doc["document_subtype"] = "Movie"
        elif doc["series_data"] is not None:
            doc["document_subtype"] = "Series"
        elif doc["episode_data"] is not None:
            doc["document_subtype"] = "Episode"
        else:
            doc["document_subtype"] = "GenericWatchItem"

        watchitem_documents.append(doc)

    # -----------------------------------------------------
    # Coleções completas do domínio
    # Mesmo que algumas não sejam usadas diretamente pela
    # query-alvo do M1, elas continuam existindo no banco
    # para manter a comparação justa.
    # -----------------------------------------------------
    user_documents = [
        row_to_python_dict(row)
        for _, row in users_df.iterrows()
    ]

    person_documents = [
        row_to_python_dict(row)
        for _, row in persons_df.iterrows()
    ]

    genre_documents = [
        row_to_python_dict(row)
        for _, row in genres_df.iterrows()
    ]

    movie_documents = [
        row_to_python_dict(row)
        for _, row in movies_df.iterrows()
    ]

    series_documents = [
        row_to_python_dict(row)
        for _, row in series_df.iterrows()
    ]

    episode_documents = [
        row_to_python_dict(row)
        for _, row in episodes_df.iterrows()
    ]

    role_documents = [
        row_to_python_dict(row)
        for _, row in roles_df.iterrows()
    ]

    rate_documents = [
        row_to_python_dict(row)
        for _, row in rates_df.iterrows()
    ]

    # -----------------------------------------------------
    # Montar o payload final FULL
    # -----------------------------------------------------
    payload = MongoExperimentPayload(
        experiment_id=experiment_row["experiment_id"],
        mongo_db_name=experiment_row["mongo_db_name"],
        selected_root=experiment_row["selected_root"],
        primary_collection=experiment_row["primary_collection"],
        collections={
            # coleção principal otimizada
            "watchitems": watchitem_documents,

            # restante do domínio
            "users": user_documents,
            "persons": person_documents,
            "genres": genre_documents,
            "movies": movie_documents,
            "series": series_documents,
            "episodes": episode_documents,
            "roles": role_documents,
            "rates": rate_documents,
        }
    )

    return payload

# ---------------------------------------------------------
# 4) Executar para o experimento ativo
# Mantemos o nome mongo_payload_m1 para não quebrar os
# blocos seguintes.
# ---------------------------------------------------------
if active_experiment_row["design_id"] != "M1":
    raise ValueError(
        "O experimento ativo atual não é M1. "
        "Para este passo, mantenha ACTIVE_EXPERIMENT_ID = 'M1_SF1'."
    )

mongo_payload_m1 = build_m1_watchitem_shallow_full_payload(
    bundle=imdb_data_bundle,
    experiment_row=active_experiment_row
)

print("Payload FULL do M1 gerado com sucesso.")
print("Banco MongoDB alvo:", mongo_payload_m1.mongo_db_name)
print("Coleção principal:", mongo_payload_m1.primary_collection)
display(mongo_payload_m1.summary())

Payload FULL do M1 gerado com sucesso.
Banco MongoDB alvo: imdb_mongo_sf1_m1_watchitem_shallow
Coleção principal: watchitems


,collection_name,n_documents,sample_top_level_keys
0,episodes,31,"[episode_number, length_min, season, series_wa..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[income, length_min, media, watchitem_id]"
3,persons,120,"[date_of_birth, gender, name, person_id]"
4,rates,547,"[rate_id, rating, user_id, verbal_rating, watc..."
5,roles,290,"[person_id, role_id, role_type, watchitem_id]"
6,series,15,"[network, seasons, watchitem_id]"
7,users,100,"[email, last_login, password, user_id, username]"
8,watchitems,80,"[avg_rating, document_subtype, episode_data, g..."


In [52]:
# =========================================================
# BLOCO V26 — INSPECIONAR O PAYLOAD FULL DO M1
# =========================================================

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "mongo_payload_m1" not in globals():
    raise NameError(
        "O objeto 'mongo_payload_m1' não está definido. Rode primeiro o bloco V25."
    )

# ---------------------------------------------------------
# 2) Resumo geral
# ---------------------------------------------------------
print("Resumo do payload FULL MongoDB M1:")
display(mongo_payload_m1.summary())

# ---------------------------------------------------------
# 3) Prévia da coleção principal
# ---------------------------------------------------------
print("Prévia: primeiro documento de watchitems")
display(pd.DataFrame([mongo_payload_m1.collections["watchitems"][0]]))

# ---------------------------------------------------------
# 4) Verificar se o domínio completo está presente
# ---------------------------------------------------------
print("Coleções presentes no payload:")
print(mongo_payload_m1.collection_names())

# ---------------------------------------------------------
# 5) Contagem por subtipo documental
# ---------------------------------------------------------
watchitems_docs_df = pd.DataFrame(mongo_payload_m1.collections["watchitems"])

print("Contagem por document_subtype:")
display(
    watchitems_docs_df["document_subtype"]
    .value_counts(dropna=False)
    .rename_axis("document_subtype")
    .reset_index(name="count")
)

# ---------------------------------------------------------
# 6) Exemplos de subtipos
# ---------------------------------------------------------
print("Exemplo de documento do tipo Movie:")
movie_docs = [
    doc for doc in mongo_payload_m1.collections["watchitems"]
    if doc["document_subtype"] == "Movie"
]
if len(movie_docs) > 0:
    display(pd.DataFrame([movie_docs[0]]))

print("Exemplo de documento do tipo Series:")
series_docs = [
    doc for doc in mongo_payload_m1.collections["watchitems"]
    if doc["document_subtype"] == "Series"
]
if len(series_docs) > 0:
    display(pd.DataFrame([series_docs[0]]))

print("Exemplo de documento do tipo Episode:")
episode_docs = [
    doc for doc in mongo_payload_m1.collections["watchitems"]
    if doc["document_subtype"] == "Episode"
]
if len(episode_docs) > 0:
    display(pd.DataFrame([episode_docs[0]]))

Resumo do payload FULL MongoDB M1:


,collection_name,n_documents,sample_top_level_keys
0,episodes,31,"[episode_number, length_min, season, series_wa..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[income, length_min, media, watchitem_id]"
3,persons,120,"[date_of_birth, gender, name, person_id]"
4,rates,547,"[rate_id, rating, user_id, verbal_rating, watc..."
5,roles,290,"[person_id, role_id, role_type, watchitem_id]"
6,series,15,"[network, seasons, watchitem_id]"
7,users,100,"[email, last_login, password, user_id, username]"
8,watchitems,80,"[avg_rating, document_subtype, episode_data, g..."


Prévia: primeiro documento de watchitems


,watchitem_id,title,release_year,avg_rating,genre_id,item_type,genre,movie_data,series_data,episode_data,document_subtype
0,1,Title 1,2013,4.66,10,Episode,"{'genre_id': 10, 'name': 'Mystery'}",None,None,"{'watchitem_id': 1, 'season': 5, 'episode_numb...",Episode


Coleções presentes no payload:
['watchitems', 'users', 'persons', 'genres', 'movies', 'series', 'episodes', 'roles', 'rates']
Contagem por document_subtype:


...

,watchitem_id,title,release_year,avg_rating,genre_id,item_type,genre,movie_data,series_data,episode_data,document_subtype
0,2,Title 2,2019,2.5,5,Movie,"{'genre_id': 5, 'name': 'SciFi'}","{'watchitem_id': 2, 'length_min': 109, 'media'...",None,None,Movie


Exemplo de documento do tipo Series:


,watchitem_id,title,release_year,avg_rating,genre_id,item_type,genre,movie_data,series_data,episode_data,document_subtype
0,10,Title 10,1994,9.66,4,Series,"{'genre_id': 4, 'name': 'Thriller'}",None,"{'watchitem_id': 10, 'seasons': 7, 'network': ...",None,Series


Exemplo de documento do tipo Episode:


,watchitem_id,title,release_year,avg_rating,genre_id,item_type,genre,movie_data,series_data,episode_data,document_subtype
0,1,Title 1,2013,4.66,10,Episode,"{'genre_id': 10, 'name': 'Mystery'}",None,None,"{'watchitem_id': 1, 'season': 5, 'episode_numb...",Episode


In [ ]:
#para conectar no servidor remoto - rodar no terminal 
    ssh -N \
   -L 57017:127.0.0.1:27018 \
   hudson@150.162.57.138

In [36]:
pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 55.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [41]:
# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL
# =========================================================

TUNNEL_HOST = "127.0.0.1"

# PostgreSQL
"""try:
    import psycopg
    conn = psycopg.connect(
        host=TUNNEL_HOST,
        port=55432,
        dbname="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    conn.close()
    print("PostgreSQL via túnel: conexão OK")
except Exception as e:
    print("PostgreSQL via túnel: falha na conexão")
    print(type(e).__name__, e)

# Cassandra
try:
    from cassandra.cluster import Cluster
    cluster = Cluster(contact_points=[TUNNEL_HOST], port=59042)
    session = cluster.connect()
    session.shutdown()
    cluster.shutdown()
    print("Cassandra via túnel: conexão OK")
except Exception as e:
    print("Cassandra via túnel: falha na conexão")
    print(type(e).__name__, e)

"""
#mongoDB

# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL — MONGODB
# =========================================================

from pymongo import MongoClient

try:
    client = MongoClient(
        host="127.0.0.1",
        port=57017,
        username="mongo",
        password="mongo",
        authSource="admin",
        serverSelectionTimeoutMS=5000
    )
    client.admin.command("ping")
    print("MongoDB via túnel: conexão OK")
    client.close()
except Exception as e:
    print("MongoDB via túnel: falha na conexão")
    print(type(e).__name__, e)

MongoDB via túnel: conexão OK


In [42]:
# =========================================================
# BLOCO V27 — CONEXÃO E UTILITÁRIOS MONGODB
# =========================================================

from pathlib import Path

try:
    from pymongo import MongoClient
    HAS_PYMONGO = True
except ImportError:
    HAS_PYMONGO = False
    MongoClient = None


# ---------------------------------------------------------
# 1) Função para abrir conexão com MongoDB
# ---------------------------------------------------------
def mongo_connect_basic(host="127.0.0.1", port=57017, username="mongo", password="mongo", auth_source="admin"):
    """
    Abre conexão com MongoDB.

    Ajuste host/port se você estiver usando:
    - Docker com porta diferente
    - túnel SSH
    - autenticação
    """
    if not HAS_PYMONGO:
        raise ImportError(
            "pymongo não está instalado. Rode: pip install pymongo"
        )

    if username and password:
        client = MongoClient(
            host=host,
            port=port,
            username=username,
            password=password,
            authSource=auth_source,
            serverSelectionTimeoutMS=5000
        )
    else:
        client = MongoClient(
            host=host,
            port=port,
            serverSelectionTimeoutMS=5000
        )

    # força teste de conexão
    client.admin.command("ping")
    return client


# ---------------------------------------------------------
# 2) Função para apagar um database inteiro, se existir
# ---------------------------------------------------------
def drop_mongo_database_if_exists(client, db_name: str):
    """
    Remove o database inteiro, se ele existir.
    Útil para recomeçar o experimento limpo.
    """
    existing_dbs = client.list_database_names()

    if db_name in existing_dbs:
        client.drop_database(db_name)
        print(f"Database removido: {db_name}")
    else:
        print(f"Database ainda não existia: {db_name}")


# ---------------------------------------------------------
# 3) Função para contar documentos de todas as coleções
# ---------------------------------------------------------
def verify_mongo_collection_counts(db, collection_names):
    """
    Retorna um DataFrame com a contagem de documentos por coleção.
    """
    rows = []

    for collection_name in collection_names:
        count = db[collection_name].count_documents({})
        rows.append({
            "collection_name": collection_name,
            "document_count": count
        })

    return pd.DataFrame(rows).sort_values("collection_name").reset_index(drop=True)

In [53]:
# =========================================================
# BLOCO V28 — CARREGAR O PAYLOAD M1 NO MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "mongo_payload_m1",
    "mongo_connect_basic",
    "drop_mongo_database_if_exists",
    "verify_mongo_collection_counts"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se sua porta do MongoDB não for 27017
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Conectar
# ---------------------------------------------------------
mongo_client = mongo_connect_basic(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE
)

print("Conexão com MongoDB: OK")

# ---------------------------------------------------------
# 4) Selecionar database do experimento
# ---------------------------------------------------------
target_db_name = mongo_payload_m1.mongo_db_name
print("Database alvo:", target_db_name)

# ---------------------------------------------------------
# 5) Apagar database anterior, se existir
# Isso garante repetibilidade do experimento.
# ---------------------------------------------------------
drop_mongo_database_if_exists(mongo_client, target_db_name)

# ---------------------------------------------------------
# 6) Criar database e carregar as coleções
# ---------------------------------------------------------
mongo_db = mongo_client[target_db_name]

for collection_name, documents in mongo_payload_m1.collections.items():
    if len(documents) > 0:
        mongo_db[collection_name].insert_many(documents)
        print(f"Coleção carregada: {collection_name} ({len(documents)} documentos)")
    else:
        print(f"Coleção vazia ignorada: {collection_name}")

print("\nCarga MongoDB finalizada.")

Conexão com MongoDB: OK
Database alvo: imdb_mongo_sf1_m1_watchitem_shallow
Database removido: imdb_mongo_sf1_m1_watchitem_shallow
Coleção carregada: watchitems (80 documentos)
Coleção carregada: users (100 documentos)
Coleção carregada: persons (120 documentos)
Coleção carregada: genres (10 documentos)
Coleção carregada: movies (34 documentos)
Coleção carregada: series (15 documentos)
Coleção carregada: episodes (31 documentos)
Coleção carregada: roles (290 documentos)
Coleção carregada: rates (547 documentos)

Carga MongoDB finalizada.


In [54]:
# =========================================================
# BLOCO V29 — CRIAR ÍNDICES PARA O M1-FULL
# =========================================================

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "O objeto 'mongo_db' não está definido. Rode primeiro o bloco V28."
    )

# ---------------------------------------------------------
# 2) Índices da coleção principal watchitems
# M1 foi pensado principalmente para Q2 e Q4
# ---------------------------------------------------------

# Busca por título (Q2)
mongo_db["watchitems"].create_index([("title", 1)])

# Filtro por gênero, tipo e ano (Q4)
mongo_db["watchitems"].create_index([
    ("genre_id", 1),
    ("item_type", 1),
    ("release_year", 1)
])

# Campo auxiliar de subtipo
mongo_db["watchitems"].create_index([("document_subtype", 1)])

# ---------------------------------------------------------
# 3) Índices das demais coleções do domínio
# Isso ajuda a manter o banco full mais realista e também
# prepara futuras consultas/benchmarks.
# ---------------------------------------------------------
mongo_db["users"].create_index([("user_id", 1)], unique=True)
mongo_db["persons"].create_index([("person_id", 1)], unique=True)
mongo_db["genres"].create_index([("genre_id", 1)], unique=True)
mongo_db["movies"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["series"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["episodes"].create_index([("watchitem_id", 1)], unique=True)
mongo_db["roles"].create_index([("role_id", 1)], unique=True)
mongo_db["rates"].create_index([("rate_id", 1)], unique=True)

# Índices auxiliares úteis
mongo_db["roles"].create_index([
    ("watchitem_id", 1),
    ("role_type", 1),
    ("person_id", 1)
])

mongo_db["rates"].create_index([
    ("user_id", 1),
    ("watchitem_id", 1)
])

print("Índices do M1-full criados com sucesso.")

Índices do M1-full criados com sucesso.


In [55]:
# =========================================================
# BLOCO V30 — VALIDAR A CARGA DO M1
# =========================================================

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "O objeto 'mongo_db' não está definido. Rode primeiro o bloco V28."
    )

# ---------------------------------------------------------
# 2) Verificar contagem por coleção
# ---------------------------------------------------------
m1_counts_df = verify_mongo_collection_counts(
    mongo_db,
    collection_names=mongo_payload_m1.collection_names()
)

print("Contagens no MongoDB para o experimento M1:")
display(m1_counts_df)

# ---------------------------------------------------------
# 3) Verificar um documento de exemplo
# ---------------------------------------------------------
sample_watchitem_doc = mongo_db["watchitems"].find_one({}, {"_id": 0})

print("Exemplo de documento salvo no MongoDB (watchitems):")
display(pd.DataFrame([sample_watchitem_doc]))

Contagens no MongoDB para o experimento M1:


,collection_name,document_count
0,episodes,31
1,genres,10
2,movies,34
3,persons,120
4,rates,547
5,roles,290
6,series,15
7,users,100
8,watchitems,80


Exemplo de documento salvo no MongoDB (watchitems):


,watchitem_id,title,release_year,avg_rating,genre_id,item_type,genre,movie_data,series_data,episode_data,document_subtype
0,1,Title 1,2013,4.66,10,Episode,"{'genre_id': 10, 'name': 'Mystery'}",None,None,"{'watchitem_id': 1, 'season': 5, 'episode_numb...",Episode


In [56]:
# =========================================================
# BLOCO V31 — CONSULTA DE TESTE DO M1
# =========================================================

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "mongo_db" not in globals():
    raise NameError(
        "O objeto 'mongo_db' não está definido. Rode primeiro o bloco V28."
    )

# ---------------------------------------------------------
# 2) Teste simples no estilo Q4
# Escolhe um documento qualquer e usa seus próprios valores
# ---------------------------------------------------------
sample_for_q4 = mongo_db["watchitems"].find_one(
    {},
    {
        "_id": 0,
        "genre_id": 1,
        "item_type": 1,
        "release_year": 1
    }
)

if sample_for_q4 is None:
    raise ValueError("Nenhum documento encontrado em watchitems.")

genre_id = sample_for_q4["genre_id"]
item_type = sample_for_q4["item_type"]
center_year = sample_for_q4["release_year"]

year_low = center_year - 10
year_high = center_year + 10

query = {
    "genre_id": genre_id,
    "item_type": item_type,
    "release_year": {
        "$gte": year_low,
        "$lte": year_high
    }
}

q4_result = list(
    mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )
)

print("Parâmetros usados no teste tipo Q4:")
print({
    "genre_id": genre_id,
    "item_type": item_type,
    "year_low": year_low,
    "year_high": year_high
})

print(f"\nNúmero de documentos retornados: {len(q4_result)}")

if len(q4_result) > 0:
    display(pd.DataFrame(q4_result[:10]))

Parâmetros usados no teste tipo Q4:
{'genre_id': 10, 'item_type': 'Episode', 'year_low': 2003, 'year_high': 2023}

Número de documentos retornados: 2


,watchitem_id,title,release_year,avg_rating,genre,document_subtype
0,1,Title 1,2013,4.66,"{'genre_id': 10, 'name': 'Mystery'}",Episode
1,61,Title 61,2017,9.75,"{'genre_id': 10, 'name': 'Mystery'}",Episode


In [57]:
# =========================================================
# BLOCO V32 — CONEXÃO DE BENCHMARK PARA O M1
# =========================================================

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "mongo_connect_basic",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Parâmetros de conexão
# Ajuste aqui se seu Mongo estiver em outra porta
# ---------------------------------------------------------
MONGO_HOST = "127.0.0.1"
MONGO_PORT = 57017
MONGO_USERNAME = "mongo"
MONGO_PASSWORD = "mongo"
MONGO_AUTH_SOURCE = "admin"

# ---------------------------------------------------------
# 3) Funções utilitárias para benchmark
# ---------------------------------------------------------
def open_mongo_benchmark_connection(
    host=MONGO_HOST,
    port=MONGO_PORT,
    username=MONGO_USERNAME,
    password=MONGO_PASSWORD,
    auth_source=MONGO_AUTH_SOURCE,
    db_name=None
):
    """
    Abre conexão Mongo e retorna:
    - client
    - db
    """
    client = mongo_connect_basic(
        host=host,
        port=port,
        username=username,
        password=password,
        auth_source=auth_source
    )

    if db_name is None:
        raise ValueError("db_name precisa ser informado.")

    db = client[db_name]
    return client, db


def close_mongo_benchmark_connection(client):
    """
    Fecha a conexão Mongo de forma limpa.
    """
    try:
        if client is not None:
            client.close()
    except Exception:
        pass

# ---------------------------------------------------------
# 4) Abrir uma conexão de teste
# ---------------------------------------------------------
mongo_bench_client, mongo_bench_db = open_mongo_benchmark_connection(
    db_name=active_experiment_row["mongo_db_name"]
)

print("Conexão de benchmark MongoDB aberta com sucesso.")
print("Database:", active_experiment_row["mongo_db_name"])

Conexão de benchmark MongoDB aberta com sucesso.
Database: imdb_mongo_sf1_m1_watchitem_shallow


In [58]:
# =========================================================
# BLOCO V33 — QUERIES MONGODB DO M1
# =========================================================

# ---------------------------------------------------------
# Q2 — SimpleSearch no M1
# Busca por título diretamente na coleção watchitems
# ---------------------------------------------------------
def run_q2_simple_search_m1_mongodb(mongo_db, title: str):
    cursor = mongo_db["watchitems"].find(
        {"title": str(title)},
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1,
            "movie_data": 1,
            "series_data": 1,
            "episode_data": 1
        }
    )
    return list(cursor)


# ---------------------------------------------------------
# Q4 — Recommendation no M1
# Filtra diretamente em watchitems por:
# - genre_id
# - item_type
# - release_year em faixa
# ---------------------------------------------------------
def run_q4_recommendation_m1_mongodb(
    mongo_db,
    genre_id: int,
    item_type: str,
    year_low: int,
    year_high: int
):
    query = {
        "genre_id": int(genre_id),
        "item_type": str(item_type),
        "release_year": {
            "$gte": int(year_low),
            "$lte": int(year_high)
        }
    }

    cursor = mongo_db["watchitems"].find(
        query,
        {
            "_id": 0,
            "watchitem_id": 1,
            "title": 1,
            "release_year": 1,
            "avg_rating": 1,
            "genre": 1,
            "document_subtype": 1
        }
    )

    return list(cursor)


# ---------------------------------------------------------
# Teste rápido das funções
# ---------------------------------------------------------
sample_title = mongo_bench_db["watchitems"].find_one({}, {"_id": 0, "title": 1})["title"]

q2_test_result = run_q2_simple_search_m1_mongodb(
    mongo_bench_db,
    title=sample_title
)

print("Teste rápido da Q2 no M1:")
print("Título usado:", sample_title)
print("Número de documentos retornados:", len(q2_test_result))

sample_q4 = mongo_bench_db["watchitems"].find_one(
    {},
    {"_id": 0, "genre_id": 1, "item_type": 1, "release_year": 1}
)

q4_test_result = run_q4_recommendation_m1_mongodb(
    mongo_bench_db,
    genre_id=sample_q4["genre_id"],
    item_type=sample_q4["item_type"],
    year_low=sample_q4["release_year"] - 10,
    year_high=sample_q4["release_year"] + 10
)

print("\nTeste rápido da Q4 no M1:")
print("Parâmetros usados:", sample_q4)
print("Número de documentos retornados:", len(q4_test_result))

Teste rápido da Q2 no M1:
Título usado: Title 1
Número de documentos retornados: 1

Teste rápido da Q4 no M1:
Parâmetros usados: {'release_year': 2013, 'genre_id': 10, 'item_type': 'Episode'}
Número de documentos retornados: 2


In [61]:
# =========================================================
# BLOCO V34B — MINI-BENCHMARK COM VALIDAÇÃO AUTOMÁTICA
# =========================================================

import time
import random
import pandas as pd

def run_m1_focal_benchmark_validated(
    mongo_db,
    parameter_pool,
    cold_repetitions=3,
    hot_repetitions=20,
    seed=42
):
    rng = random.Random(seed)
    results = []

    def append_result(query_name, phase, run_id, start_time, success, error_message):
        latency_ms = (time.perf_counter() - start_time) * 1000.0
        results.append({
            "experiment_id": active_experiment_row["experiment_id"],
            "mongo_db_name": active_experiment_row["mongo_db_name"],
            "query_name": query_name,
            "benchmark_phase": phase,
            "run_id": run_id,
            "latency_ms": latency_ms,
            "success": success,
            "error_message": error_message
        })

    # -----------------------------
    # COLD RUNS — Q2
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q2_simple_search_m1_mongodb(local_db, title)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q2_SimpleSearch", "cold", run_id, start, success, error_message)

    # -----------------------------
    # COLD RUNS — Q4
    # -----------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        local_client = None
        local_db = None
        try:
            local_client, local_db = open_mongo_benchmark_connection(
                db_name=active_experiment_row["mongo_db_name"]
            )
            _ = run_q4_recommendation_m1_mongodb(
                local_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_mongo_benchmark_connection(local_client)

        append_result("Q4_Recommendation", "cold", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q2
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        title = rng.choice(parameter_pool["Q2_SimpleSearch"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q2_simple_search_m1_mongodb(mongo_db, title)
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q2_SimpleSearch", "hot", run_id, start, success, error_message)

    # -----------------------------
    # HOT RUNS — Q4
    # -----------------------------
    for run_id in range(1, hot_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        start = time.perf_counter()
        success = True
        error_message = None

        try:
            _ = run_q4_recommendation_m1_mongodb(
                mongo_db,
                genre_id=row["genre_id"],
                item_type=row["item_type"],
                year_low=row["release_year"] - 10,
                year_high=row["release_year"] + 10
            )
        except Exception as e:
            success = False
            error_message = str(e)

        append_result("Q4_Recommendation", "hot", run_id, start, success, error_message)

    benchmark_df = pd.DataFrame(results)

    # -----------------------------
    # VALIDAÇÃO AUTOMÁTICA
    # -----------------------------
    expected_counts = {
        ("Q2_SimpleSearch", "cold"): cold_repetitions,
        ("Q2_SimpleSearch", "hot"): hot_repetitions,
        ("Q4_Recommendation", "cold"): cold_repetitions,
        ("Q4_Recommendation", "hot"): hot_repetitions,
    }

    actual_counts = (
        benchmark_df
        .groupby(["query_name", "benchmark_phase"])
        .size()
        .to_dict()
    )

    for key, expected_value in expected_counts.items():
        actual_value = actual_counts.get(key, 0)
        if actual_value != expected_value:
            raise ValueError(
                f"Contagem inesperada para {key}: esperado={expected_value}, obtido={actual_value}"
            )

    return benchmark_df

In [62]:
# =========================================================
# BLOCO V34A — DIAGNÓSTICO E EXIBIÇÃO CORRETA DO MINI-BENCHMARK
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Checagem básica
# ---------------------------------------------------------
if "m1_benchmark_df" not in globals():
    raise NameError(
        "O DataFrame 'm1_benchmark_df' não está definido. "
        "Rode primeiro o bloco do mini-benchmark."
    )

# ---------------------------------------------------------
# 2) Ordenar explicitamente o resultado
# Isso ajuda a visualizar tudo de forma consistente
# ---------------------------------------------------------
m1_benchmark_df = m1_benchmark_df.sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 3) Contagem de execuções por query e fase
# Esse é o teste mais importante para saber se Q4 hot existe
# ---------------------------------------------------------
m1_counts_by_query_phase_df = (
    m1_benchmark_df
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        n_runs=("run_id", "count"),
        n_success=("success", lambda s: int(s.sum())),
        n_failures=("success", lambda s: int((~s).sum()))
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Contagem de execuções por query e fase:")
display(m1_counts_by_query_phase_df)

# ---------------------------------------------------------
# 4) Mostrar explicitamente as linhas de Q4 hot
# ---------------------------------------------------------
m1_q4_hot_df = m1_benchmark_df[
    (m1_benchmark_df["query_name"] == "Q4_Recommendation") &
    (m1_benchmark_df["benchmark_phase"] == "hot")
].copy()

print("Linhas de Q4 hot:")
display(m1_q4_hot_df)

# ---------------------------------------------------------
# 5) Resumo estatístico simples por query e fase
# ---------------------------------------------------------
m1_summary_df = (
    m1_benchmark_df[m1_benchmark_df["success"] == True]
    .groupby(["query_name", "benchmark_phase"], as_index=False)
    .agg(
        runs=("latency_ms", "count"),
        avg_ms=("latency_ms", "mean"),
        median_ms=("latency_ms", "median"),
        min_ms=("latency_ms", "min"),
        max_ms=("latency_ms", "max")
    )
    .sort_values(["query_name", "benchmark_phase"])
    .reset_index(drop=True)
)

print("Resumo estatístico do mini-benchmark:")
display(m1_summary_df)

# ---------------------------------------------------------
# 6) Mostrar tudo, se quiser inspecionar linha a linha
# ---------------------------------------------------------
print("Resultados brutos completos do mini-benchmark:")
display(m1_benchmark_df)

# ---------------------------------------------------------
# 7) Falhas, se existirem
# ---------------------------------------------------------
m1_failures_df = m1_benchmark_df[m1_benchmark_df["success"] == False].copy()

print("Falhas, se existirem:")
display(m1_failures_df)

Contagem de execuções por query e fase:


,query_name,benchmark_phase,n_runs,n_success,n_failures
0,Q2_SimpleSearch,cold,3,3,0
1,Q2_SimpleSearch,hot,20,20,0
2,Q4_Recommendation,cold,3,3,0
3,Q4_Recommendation,hot,20,20,0


Linhas de Q4 hot:


,experiment_id,mongo_db_name,query_name,benchmark_phase,run_id,latency_ms,success,error_message
26,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,1,1.767581,True,None
27,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,2,1.581439,True,None
28,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,3,1.295116,True,None
29,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,4,1.689080,True,None
30,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,5,2.401079,True,None
31,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,6,3.363911,True,None
32,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,7,1.905302,True,None
33,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,8,1.449617,True,None
34,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,9,1.663680,True,None
35,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q4_Recommendation,hot,10,1.740981,True,None


Resumo estatístico do mini-benchmark:


,query_name,benchmark_phase,runs,avg_ms,median_ms,min_ms,max_ms
0,Q2_SimpleSearch,cold,3,62.655722,63.111237,61.730432,63.125497
1,Q2_SimpleSearch,hot,20,1.356818,1.304111,1.062073,2.169516
2,Q4_Recommendation,cold,3,65.507393,66.412377,63.315800,66.794002
3,Q4_Recommendation,hot,20,1.633833,1.450428,1.136773,3.363911


Resultados brutos completos do mini-benchmark:


,experiment_id,mongo_db_name,query_name,benchmark_phase,run_id,latency_ms,success,error_message
0,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,cold,1,61.730432,True,None
1,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,cold,2,63.111237,True,None
2,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,cold,3,63.125497,True,None
3,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,1,2.169516,True,None
4,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,2,1.646269,True,None
5,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,3,1.721641,True,None
6,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,4,1.364667,True,None
7,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,5,1.350806,True,None
8,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,6,1.200784,True,None
9,M1_SF1,imdb_mongo_sf1_m1_watchitem_shallow,Q2_SimpleSearch,hot,7,1.328726,True,None


Falhas, se existirem:


,experiment_id,mongo_db_name,query_name,benchmark_phase,run_id,latency_ms,success,error_message


In [63]:
# =========================================================
# BLOCO V35 — EXPORTAR RESULTADOS DO BENCHMARK POR EXPERIMENTO
# =========================================================

from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# 1) Checagens básicas
# ---------------------------------------------------------
required_names = [
    "m1_benchmark_df",
    "active_experiment_row"
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Função para resumir estatísticas básicas
# ---------------------------------------------------------
def summarize_benchmark_basic(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um resumo simples por query e fase.
    """
    if result_df.empty:
        return pd.DataFrame()

    ok_df = result_df[result_df["success"] == True].copy()

    if ok_df.empty:
        return pd.DataFrame()

    summary_df = (
        ok_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            runs=("latency_ms", "count"),
            avg_ms=("latency_ms", "mean"),
            median_ms=("latency_ms", "median"),
            min_ms=("latency_ms", "min"),
            max_ms=("latency_ms", "max"),
            std_ms=("latency_ms", "std")
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    summary_df["std_ms"] = summary_df["std_ms"].fillna(0.0)
    return summary_df


# ---------------------------------------------------------
# 3) Função para contar execuções por query e fase
# ---------------------------------------------------------
def build_counts_by_query_phase(result_df: pd.DataFrame) -> pd.DataFrame:
    """
    Conta quantas execuções ocorreram por query e fase,
    incluindo sucessos e falhas.
    """
    if result_df.empty:
        return pd.DataFrame()

    counts_df = (
        result_df
        .groupby(["query_name", "benchmark_phase"], as_index=False)
        .agg(
            n_runs=("run_id", "count"),
            n_success=("success", lambda s: int(s.sum())),
            n_failures=("success", lambda s: int((~s).sum()))
        )
        .sort_values(["query_name", "benchmark_phase"])
        .reset_index(drop=True)
    )

    return counts_df


# ---------------------------------------------------------
# 4) Definir diretório do experimento
# Cada experimento ganha sua própria pasta
# ---------------------------------------------------------
experiment_id = active_experiment_row["experiment_id"]

experiment_results_dir = Path("exports/mongodb_benchmarks") / experiment_id
experiment_results_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 5) Preparar DataFrames para exportação
# ---------------------------------------------------------
benchmark_raw_df = m1_benchmark_df.copy().sort_values(
    by=["query_name", "benchmark_phase", "run_id"]
).reset_index(drop=True)

benchmark_summary_df = summarize_benchmark_basic(benchmark_raw_df)

benchmark_failures_df = benchmark_raw_df[
    benchmark_raw_df["success"] == False
].copy().reset_index(drop=True)

benchmark_counts_df = build_counts_by_query_phase(benchmark_raw_df)

# ---------------------------------------------------------
# 6) Definir nomes dos arquivos com prefixo do experimento
# ---------------------------------------------------------
raw_csv_path = experiment_results_dir / f"{experiment_id}_raw.csv"
summary_csv_path = experiment_results_dir / f"{experiment_id}_summary.csv"
failures_csv_path = experiment_results_dir / f"{experiment_id}_failures.csv"
counts_csv_path = experiment_results_dir / f"{experiment_id}_counts_by_query_phase.csv"

# ---------------------------------------------------------
# 7) Salvar os CSVs
# ---------------------------------------------------------
benchmark_raw_df.to_csv(raw_csv_path, index=False)
benchmark_summary_df.to_csv(summary_csv_path, index=False)
benchmark_failures_df.to_csv(failures_csv_path, index=False)
benchmark_counts_df.to_csv(counts_csv_path, index=False)

# ---------------------------------------------------------
# 8) Mostrar os caminhos gerados
# ---------------------------------------------------------
print("Arquivos CSV do benchmark exportados com sucesso:")
print(raw_csv_path)
print(summary_csv_path)
print(failures_csv_path)
print(counts_csv_path)

# ---------------------------------------------------------
# 9) Mostrar prévia dos dados exportados
# ---------------------------------------------------------
print("\nResumo estatístico exportado:")
display(benchmark_summary_df)

print("Contagem por query e fase exportada:")
display(benchmark_counts_df)

print("Falhas exportadas:")
display(benchmark_failures_df)

Arquivos CSV do benchmark exportados com sucesso:
exports/mongodb_benchmarks/M1_SF1/M1_SF1_raw.csv
exports/mongodb_benchmarks/M1_SF1/M1_SF1_summary.csv
exports/mongodb_benchmarks/M1_SF1/M1_SF1_failures.csv
exports/mongodb_benchmarks/M1_SF1/M1_SF1_counts_by_query_phase.csv

Resumo estatístico exportado:


,query_name,benchmark_phase,runs,avg_ms,median_ms,min_ms,max_ms,std_ms
0,Q2_SimpleSearch,cold,3,62.655722,63.111237,61.730432,63.125497,0.801356
1,Q2_SimpleSearch,hot,20,1.356818,1.304111,1.062073,2.169516,0.266733
2,Q4_Recommendation,cold,3,65.507393,66.412377,63.315800,66.794002,1.907543
3,Q4_Recommendation,hot,20,1.633833,1.450428,1.136773,3.363911,0.530933


Contagem por query e fase exportada:


,query_name,benchmark_phase,n_runs,n_success,n_failures
0,Q2_SimpleSearch,cold,3,3,0
1,Q2_SimpleSearch,hot,20,20,0
2,Q4_Recommendation,cold,3,3,0
3,Q4_Recommendation,hot,20,20,0


Falhas exportadas:


,experiment_id,mongo_db_name,query_name,benchmark_phase,run_id,latency_ms,success,error_message
